# Atlas v3.2 | ShadowBytes
Auditoría retrospectiva. Ningún submit automático. El modo full reconstruye Ultralisk v2.7 en el nuevo corte. Usar directorios nuevos.

In [ ]:
SOURCE='ATLAS_SOURCE=\'"""Atlas v3.1: independent residual physical downscaling."""\\nimport os\\nfor k in (\\\'OPENBLAS_NUM_THREADS\\\',\\\'OMP_NUM_THREADS\\\',\\\'MKL_NUM_THREADS\\\'): os.environ[k]=\\\'4\\\'\\nimport io, zipfile, time, argparse\\nfrom pathlib import Path\\nimport numpy as np\\nimport pandas as pd\\nfrom scipy.linalg import solve\\nfrom sklearn.utils.extmath import randomized_svd\\nfrom threadpoolctl import threadpool_limits\\nNORM=np.array([25.360,2.0652,2.6776,2.0876],dtype=\\\'f4\\\')\\nCUTS=[(\\\'Mayo\\\',\\\'2024050900\\\',\\\'2024051600\\\',\\\'2024060100\\\'),(\\\'Junio I\\\',\\\'2024052500\\\',\\\'2024060100\\\',\\\'2024061600\\\'),(\\\'Junio II\\\',\\\'2024060900\\\',\\\'2024061600\\\',\\\'2024070100\\\'),(\\\'Abril\\\',\\\'2024040900\\\',\\\'2024041600\\\',\\\'2024050100\\\')]\\ndef load(root):\\n    root=Path(root)\\n    if root.is_file():\\n        with zipfile.ZipFile(root) as z:\\n            def read(n):\\n                with np.load(io.BytesIO(z.read(n))) as f: return dict(f)\\n            return read(\\\'train.npz\\\'),read(\\\'static.npz\\\')\\n    if not (root/\\\'train.npz\\\').exists():\\n        matches=[p.parent for p in root.rglob(\\\'train.npz\\\') if (p.parent/\\\'static.npz\\\').exists() and (p.parent/\\\'test.npz\\\').exists()]\\n        if len(matches)!=1: raise ValueError(f\\\'Dataset ambiguo: {matches}\\\')\\n        root=matches[0]\\n    def read(n):\\n        with np.load(root/n) as f: return dict(f)\\n    return read(\\\'train.npz\\\'),read(\\\'static.npz\\\')\\ndef windows(x,t):\\n    dates=pd.to_datetime(t,format=\\\'%Y%m%d%H\\\'); lookup={d:i for i,d in enumerate(dates)}\\n    wanted=[[d-pd.Timedelta(hours=k) for k in (2,1,0)] for d in dates]\\n    idx=np.array([[lookup.get(q,i) for q in row] for i,row in enumerate(wanted)])\\n    mask=np.array([[q in lookup for q in row] for row in wanted],dtype=\\\'f4\\\')\\n    return x[idx],mask\\ndef fields(w,lat,lon):\\n    rh,t,u,v=np.moveaxis(w,-1,0); c=t-273.15\\n    gamma=np.log(np.clip(rh,.1,100)/100)+17.625*c/(243.04+c)\\n    dew=243.04*gamma/(17.625-gamma)\\n    e=6.1094*np.exp(17.625*c/(243.04+c))*rh/100\\n    dy=np.deg2rad(float(np.diff(lat).mean()))*6371000\\n    dx=np.deg2rad(float(np.diff(lon).mean()))*6371000*np.cos(np.deg2rad(lat))[None,None,:,None]\\n    def grad(a): return np.gradient(a,axis=-1)/dx,np.gradient(a,axis=-2)/dy\\n    ux,uy=grad(u); vx,vy=grad(v); tx,ty=grad(t); ex,ey=grad(e)\\n    return np.stack([rh,t,u,v,c-dew,e,np.hypot(u,v),(ux+vy)*1e5,(vx-uy)*1e5,(ux-vy)*1e5,(vx+uy)*1e5,-3600*(u*tx+v*ty),-3600*(u*ex+v*ey),np.hypot(tx,ty)*1e5,np.hypot(ex,ey)*1e5],axis=-1).astype(\\\'f4\\\')\\ndef features(w,physics,lat,lon):\\n    a=fields(w,lat,lon) if physics else w\\n    return np.concatenate([a[:,2],a[:,2]-a[:,1],a[:,2]-a[:,0]],axis=-1).reshape(len(w),-1)\\ndef climate(a,h,tr):\\n    return np.stack([a[tr[h[tr]==i]].mean(0) if np.any(h[tr]==i) else a[tr].mean(0) for i in range(24)])\\ndef metric(p,y):\\n    p=p.copy(); p[...,0]=p[...,0].clip(0,100)\\n    rm=np.sqrt(np.mean((p.astype(\\\'f8\\\')-y)**2,axis=(0,1,2)))\\n    return float(1-(rm/NORM).mean()),rm\\n\\nimport math, json, copy\\nfrom concurrent.futures import ThreadPoolExecutor\\nimport torch\\nfrom torch import nn\\nfrom torch.nn import functional as F\\n\\nBASE={\\\'Mayo\\\':.4599006,\\\'Junio I\\\':.4604175,\\\'Junio II\\\':.4650797440837502,\\\'Abril\\\':.5584575773496497}\\n\\ndef geography(s):\\n    lat,lon=np.meshgrid(s[\\\'wrf_lats\\\'],s[\\\'wrf_lons\\\'],indexing=\\\'ij\\\'); dem=s[\\\'dem\\\'].astype(\\\'f4\\\')\\n    dy=np.deg2rad(float(np.diff(s[\\\'wrf_lats\\\']).mean()))*6371000\\n    dx=np.deg2rad(float(np.diff(s[\\\'wrf_lons\\\']).mean()))*6371000*np.cos(np.deg2rad(lat))\\n    sx=np.gradient(dem,axis=1)/dx; sy=np.gradient(dem,axis=0)/dy\\n    from scipy.ndimage import gaussian_filter\\n    relief=[(dem-gaussian_filter(dem,k))/500 for k in (1,3,7)]\\n    curvature=(np.gradient(sx,axis=1)/dx+np.gradient(sy,axis=0)/dy)*1000\\n    # Raw slopes retained separately for physically interpretable wind projection.\\n    static=np.stack([dem/3000,sx,sy,np.hypot(sx,sy),*relief,curvature,(lat-lat.mean()),(lon-lon.mean())]).astype(\\\'f4\\\')\\n    gx=2*(lon-s[\\\'gfs_lons\\\'][0])/(s[\\\'gfs_lons\\\'][-1]-s[\\\'gfs_lons\\\'][0])-1\\n    gy=2*(lat-s[\\\'gfs_lats\\\'][0])/(s[\\\'gfs_lats\\\'][-1]-s[\\\'gfs_lats\\\'][0])-1\\n    grid=np.stack([gx,gy],-1)[None].astype(\\\'f4\\\')\\n    return static[None],grid\\n\\ndef prepare(X,T,s):\\n    w,mask=windows(X,T); d=fields(w,s[\\\'gfs_lats\\\'],s[\\\'gfs_lons\\\'])\\n    x=np.concatenate([d[:,2],d[:,1],d[:,0]],axis=-1).transpose(0,3,1,2).copy()\\n    dates=pd.to_datetime(T,format=\\\'%Y%m%d%H\\\'); h=dates.hour.to_numpy(); day=dates.dayofyear.to_numpy()\\n    a=h*2*np.pi/24; decl=.409*np.sin(2*np.pi*(day-81)/365.25)\\n    # Solar declination and UTC hour; longitude is applied inside the decoder.\\n    t=np.column_stack([np.sin(a),np.cos(a),np.sin(2*a),np.cos(2*a),np.sin(decl),np.cos(decl),mask]).astype(\\\'f4\\\')\\n    return x,t,h\\n\\nclass Block(nn.Module):\\n    def __init__(self,c,d=1):\\n        super().__init__(); self.f=nn.Sequential(nn.Conv2d(c,c,3,padding=d,dilation=d),nn.GroupNorm(8,c),nn.SiLU(),nn.Conv2d(c,c,3,padding=1),nn.GroupNorm(8,c))\\n    def forward(self,x): return F.silu(x+.5*self.f(x))\\n\\nclass ConvLSTM(nn.Module):\\n    def __init__(self,w):\\n        super().__init__(); self.gates=nn.Conv2d(2*w,4*w,3,padding=1)\\n    def forward(self,x,h,c):\\n        i,f,o,g=self.gates(torch.cat([x,h],1)).chunk(4,1)\\n        c=(f+1).sigmoid()*c+i.sigmoid()*g.tanh(); return o.sigmoid()*c.tanh(),c\\n\\nclass TerrainFlow(nn.Module):\\n    def __init__(self,static,grid,lat,lon,mu,sd,width=48,physics=True):\\n        super().__init__(); self.physics=physics; self.width=width\\n        for name,a in [(\\\'geo\\\',static),(\\\'grid\\\',grid),(\\\'latitude\\\',np.deg2rad(lat)[None,None,:,None]),(\\\'longitude\\\',np.deg2rad(lon)[None,None,None,:]),(\\\'mu\\\',mu[None,:4,None,None]),(\\\'sd\\\',sd[None,:4,None,None])]:\\n            self.register_buffer(name,torch.from_numpy(np.asarray(a,dtype=\\\'f4\\\')))\\n        self.stem=nn.Sequential(nn.Conv2d(24,width,3,padding=1),nn.GroupNorm(8,width),nn.SiLU()); self.temporal=ConvLSTM(width); self.context=Block(width,2)\\n        self.pos=nn.Parameter(torch.zeros(1,17*13,width)); nn.init.normal_(self.pos,std=.02)\\n        self.attn=nn.MultiheadAttention(width,4,batch_first=True,dropout=.05); self.norm=nn.LayerNorm(width)\\n        self.geoenc=nn.Sequential(nn.Conv2d(10,24,3,padding=1),nn.SiLU(),nn.Conv2d(24,24,3,padding=1),nn.SiLU())\\n        # Static terrain-conditioned experts, high-resolution local GFS and dynamic physical interactions.\\n        fine=16 if width<=24 else 32\\n        self.fuse=nn.Sequential(nn.Conv2d(width+45+24+6,fine,1),nn.GroupNorm(4,fine),nn.SiLU(),nn.Conv2d(fine,fine,3,padding=1,groups=4),nn.SiLU())\\n        self.thermo=nn.Sequential(nn.Conv2d(fine,fine,1),nn.SiLU(),nn.Conv2d(fine,2,1))\\n        self.wind=nn.Sequential(nn.Conv2d(fine,fine,1),nn.SiLU(),nn.Conv2d(fine,2,1))\\n    def up(self,x):\\n        with torch.autocast(device_type=x.device.type,enabled=False):\\n            return F.grid_sample(x.float(),self.grid.expand(len(x),-1,-1,-1),align_corners=True,padding_mode=\\\'border\\\')\\n    def forward(self,x,t):\\n        n=len(x)\\n        if not self.physics:\\n            x=x.reshape(n,3,15,17,13); x=torch.cat([x[:,:,:4],torch.zeros_like(x[:,:,4:])],2).reshape(n,45,17,13)\\n        h=x.new_zeros(n,self.width,17,13); c=h\\n        for k in (2,1,0):\\n            z=self.stem(torch.cat([x[:,k*15:(k+1)*15],t[:,:,None,None].expand(-1,-1,17,13)],1))\\n            hn,cn=self.temporal(z,h,c); valid=t[:,8-k,None,None,None]; h=valid*hn+(1-valid)*h; c=valid*cn+(1-valid)*c\\n        low=self.context(h); seq=low.flatten(2).transpose(1,2)\\n        z=self.norm(seq+self.pos); a,_=self.attn(z,z,z,need_weights=False); ctx=(seq+a).transpose(1,2).reshape_as(low)\\n        raw=self.up(x); uv=raw[:,2:4]*self.sd[:,2:4]+self.mu[:,2:4]; u,v=uv[:,0:1],uv[:,1:2]\\n        sx,sy=self.geo[:,1:2],self.geo[:,2:3]; slope=torch.sqrt(sx*sx+sy*sy+1e-6)\\n        upslope=(u*sx+v*sy)/slope; along=(-u*sy+v*sx)/slope\\n        sh,ch,sd,cd=[t[:,i:i+1,None,None] for i in (0,1,4,5)]\\n        # Solar hour angle = UTC phase + longitude - pi; neglect equation of time.\\n        cosh=-(ch*torch.cos(self.longitude)-sh*torch.sin(self.longitude))\\n        sinh=-(sh*torch.cos(self.longitude)+ch*torch.sin(self.longitude))\\n        east=-cd*sinh; north=torch.cos(self.latitude)*sd-torch.sin(self.latitude)*cd*cosh\\n        up=torch.sin(self.latitude)*sd+torch.cos(self.latitude)*cd*cosh\\n        incidence=((up-sx*east-sy*north)/torch.sqrt(1+sx*sx+sy*sy)).clamp(0,1)*(up>0)\\n        dyn=torch.cat([upslope/5,along/5,(u*sx+v*sy)/2,up.clamp(0,1).expand(-1,-1,70,35),incidence,torch.sqrt(u*u+v*v+1e-6)/5],1)\\n        if not self.physics: dyn=torch.zeros_like(dyn)\\n        geo=self.geoenc(self.geo).expand(n,-1,-1,-1); h=self.fuse(torch.cat([self.up(ctx),raw,geo,dyn],1))\\n        return torch.cat([self.thermo(h),self.wind(h)],1).permute(0,2,3,1)\\n\\nimport hashlib, threading\\nfrom scipy.linalg import cho_factor, cho_solve\\nfrom sklearn.decomposition import PCA\\nLOCK=threading.Lock(); LOG=None\\n\\ndef log(*args):\\n    with LOCK:\\n        msg=\\\' \\\'.join(map(str,args)); print(msg,flush=True)\\n        if LOG is not None:\\n            with LOG.open(\\\'a\\\') as f: f.write(msg+\\\'\\\\n\\\')\\n\\ndef load_all(root):\\n    root=Path(root); names=[\\\'train.npz\\\',\\\'test.npz\\\',\\\'static.npz\\\',\\\'sample_submission.csv\\\']\\n    if root.is_file():\\n        with zipfile.ZipFile(root) as z:\\n            data=[]\\n            for n in names[:3]:\\n                with np.load(io.BytesIO(z.read(n))) as f: data.append(dict(f))\\n            return (*data,pd.read_csv(io.BytesIO(z.read(names[3])),dtype={\\\'id\\\':str}))\\n    if not all((root/n).exists() for n in names):\\n        roots=[p.parent for p in root.rglob(\\\'train.npz\\\') if all((p.parent/n).exists() for n in names)]\\n        if len(roots)!=1: raise ValueError(f\\\'Dataset ambiguo: {roots}\\\')\\n        root=roots[0]\\n    data=[]\\n    for n in names[:3]:\\n        with np.load(root/n) as f: data.append(dict(f))\\n    return (*data,pd.read_csv(root/names[3],dtype={\\\'id\\\':str}))\\n\\ndef anchor(X,t,h,Y,tr,ev,s,alpha=.08):\\n    """Train-only local ridge, atmospheric EOFs, and hourly anomalies."""\\n    start=time.perf_counter(); n=len(tr); H,W=Y.shape[1:3]\\n    a=X[:,:15]; b=(a-X[:,15:30])*t[:,7,None,None,None]; c=(a-X[:,30:45])*t[:,6,None,None,None]\\n    z=np.concatenate([a,b[:,:7],c[:,:7]],1).transpose(0,2,3,1).copy()\\n    z-=climate(z,h,tr)[h]; z/=np.maximum(z[tr].std((0,1,2)),1e-4); z=np.clip(z,-12,12)\\n    flat=z.reshape(len(z),-1); pca=PCA(n_components=min(24,n-1),svd_solver=\\\'randomized\\\',random_state=17); pca.fit(flat[tr]); glob=pca.transform(flat); glob/=np.maximum(glob[tr].std(0),1e-4)\\n    yc=climate(Y,h,tr); target=((Y[tr]-yc[h[tr]])/NORM).reshape(n,-1,4)\\n    lat,lon=np.meshgrid(s[\\\'wrf_lats\\\'],s[\\\'wrf_lons\\\'],indexing=\\\'ij\\\'); iy=np.abs(lat.ravel()[:,None]-s[\\\'gfs_lats\\\']).argmin(1); ix=np.abs(lon.ravel()[:,None]-s[\\\'gfs_lons\\\']).argmin(1)\\n    groups=iy*13+ix; both=np.r_[tr,ev]; pred=np.empty((len(both),H*W,4),dtype=\\\'f4\\\')\\n    for g in np.unique(groups):\\n        r,c=divmod(int(g),13); pix=np.where(groups==g)[0]\\n        cells=sorted(set((int(np.clip(r+dr,0,16)),int(np.clip(c+dc,0,12))) for dr,dc in [(0,0),(-1,0),(1,0),(0,-1),(0,1)]))\\n        d=np.concatenate([z[both,r,c],*[z[both,rr,cc,:7] for rr,cc in cells if (rr,cc)!=(r,c)],glob[both],t[both,:4],t[both,6:]],1).astype(\\\'f8\\\')\\n        d=(d-d[:n].mean(0))/np.maximum(d[:n].std(0),.05); d=np.column_stack([np.ones(len(d)),d]); A=d[:n]; yy=target[:,pix].reshape(n,-1).astype(\\\'f8\\\')\\n        gram=A.T@A; reg=np.full(len(gram),alpha*n); reg[0]=1e-6; gram.flat[::len(gram)+1]+=reg\\n        beta=cho_solve(cho_factor(gram),A.T@yy); pred[:,pix]=(d@beta).reshape(len(both),len(pix),4)\\n    pred=pred.reshape(-1,H,W,4)*NORM+yc[h[both]]; pred[...,0]=pred[...,0].clip(0,100)\\n    assert np.isfinite(pred).all(); log(\\\'Anchor\\\',len(np.unique(groups)),\\\'regions\\\',round(time.perf_counter()-start,2),\\\'s\\\'); return pred[:n],pred[n:]\\n\\nclass AtlasResidual(TerrainFlow):\\n    def __init__(self,*a,**kw):\\n        super().__init__(*a,**kw)\\n        for layer in [self.thermo[-1],self.wind[-1]]:\\n            nn.init.zeros_(layer.weight); nn.init.zeros_(layer.bias)\\n\\ndef auxiliary(p,y,a):\\n    norm=p.new_tensor(NORM); p0=a+p*norm; y0=a+y*norm\\n    def vapor(x):\\n        c=(x[...,1]-273.15).clamp(-60,55); return 6.1094*torch.exp(17.625*c/(243.04+c))*x[...,0].clamp(0,100)/100\\n    speed=(torch.sqrt(p0[...,2:].square().sum(-1)+1e-5)-torch.sqrt(y0[...,2:].square().sum(-1)+1e-5))/2.5\\n    err=p-y\\n    return .02*speed.square().mean()+.015*((vapor(p0)-vapor(y0))/4).square().mean()+.01*(err.diff(dim=1).square().mean()+err.diff(dim=2).square().mean())\\n\\ndef fit(X,t,Y,tr,ev,at,ae,s,device,seed,epochs,kind,smoke,checkpoint=None):\\n    start=time.perf_counter(); mu=np.tile(X[tr,:15].mean((0,2,3)),3); sd=np.tile(np.maximum(X[tr,:15].std((0,2,3)),1e-4),3); geo,grid=geography(s)\\n    with LOCK:\\n        torch.manual_seed(seed); model=AtlasResidual(geo,grid,s[\\\'wrf_lats\\\'],s[\\\'wrf_lons\\\'],mu,sd,width=16 if smoke else 48,physics=kind!=\\\'RawResidual\\\')\\n    model=model.to(device); ema=copy.deepcopy(model).eval(); ema.requires_grad_(False); rng=np.random.default_rng(seed); batch=16 if smoke else 32; amp=device.type==\\\'cuda\\\'\\n    xa=torch.as_tensor((X[tr]-mu[None,:,None,None])/sd[None,:,None,None],device=device); ta=torch.as_tensor(t[tr],device=device); aa=torch.as_tensor(at,device=device); ya=torch.as_tensor((Y[tr]-at)/NORM,device=device)\\n    opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.001); sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=math.ceil(len(tr)/batch)*epochs,eta_min=2e-5); scaler=torch.amp.GradScaler(\\\'cuda\\\',enabled=amp); updates=0\\n    log(kind,device,seed,\\\'params\\\',sum(p.numel() for p in model.parameters()))\\n    for ep in range(epochs):\\n        model.train(); total=0.; perm=rng.permutation(len(tr))\\n        for off in range(0,len(tr),batch):\\n            ids=torch.as_tensor(perm[off:off+batch],device=device); opt.zero_grad(set_to_none=True)\\n            with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp): p=model(xa[ids],ta[ids])\\n            p=p.float(); mse=(p-ya[ids]).square().mean((0,1,2)); loss=.6*torch.sqrt(mse+1e-6).mean()+.4*mse.mean()\\n            if kind==\\\'PhysicalResidual\\\': loss=loss+auxiliary(p,ya[ids],aa[ids])\\n            if kind==\\\'WindResidual\\\':\\n                ww=p.new_tensor([.35,.65]); loss=(ww*(.6*torch.sqrt(mse[2:]+1e-6)+.4*mse[2:])).sum()\\n                err=(p-ya[ids])[...,2:]; loss=loss+.01*(err.diff(dim=1).square().mean()+err.diff(dim=2).square().mean())\\n            if not torch.isfinite(loss): raise FloatingPointError(\\\'Loss no finita\\\')\\n            scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(),1.); prev=scaler.get_scale(); scaler.step(opt); scaler.update()\\n            if scaler.get_scale()>=prev:\\n                sched.step(); updates+=1\\n                with torch.no_grad(): torch._foreach_lerp_(list(ema.parameters()),list(model.parameters()),1-min(.995,(1+updates)/(10+updates)))\\n            total+=float(loss.detach())*len(ids)\\n        log(kind,device,seed,f\\\'ep={ep+1}/{epochs} loss={total/len(tr):.6f} lr={opt.param_groups[0]["lr"]:.6f} elapsed={time.perf_counter()-start:.1f}s\\\')\\n    preds=[]; ema.eval()\\n    with torch.inference_mode():\\n        for k in range(0,len(ev),batch):\\n            idx=ev[k:k+batch]; xb=torch.as_tensor((X[idx]-mu[None,:,None,None])/sd[None,:,None,None],device=device); tb=torch.as_tensor(t[idx],device=device)\\n            with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp): p=ema(xb,tb)\\n            preds.append(p.float().cpu().numpy())\\n    pred=np.concatenate(preds)*NORM+ae; pred[...,0]=pred[...,0].clip(0,100); assert np.isfinite(pred).all()\\n    if checkpoint is not None: torch.save(dict(state_dict=ema.cpu().state_dict(),mu=mu,sd=sd,kind=kind,seed=seed),checkpoint)\\n    return pred\\n\\ndef ensemble(X,t,Y,tr,ev,at,ae,s,devices,epochs,kind,smoke,out=None):\\n    seeds=[42] if smoke else [42,123]\\n    with ThreadPoolExecutor(max_workers=len(devices)) as pool:\\n        fs=[pool.submit(fit,X,t,Y,tr,ev,at,ae,s,devices[i%len(devices)],seed,epochs,kind,smoke,None if out is None else out/f\\\'{kind}_{seed}.pt\\\') for i,seed in enumerate(seeds)]\\n        return np.mean([f.result() for f in fs],axis=0)\\n\\ndef weights(a,b,y):\\n    d=(b-a).astype(\\\'f8\\\'); r=(y-a).astype(\\\'f8\\\'); axes=(0,1,2); return (.85*np.clip((d*r).sum(axes)/np.maximum((d*d).sum(axes),1e-10),0,1)).astype(\\\'f4\\\')\\n\\ndef blend(a,b,w):\\n    p=a+(b-a)*w; p[...,0]=p[...,0].clip(0,100); return p\\n\\ndef save_csv(template,times,p,dest):\\n    assert p.shape==(len(times),70,35,4)\\n    ids=[f\\\'{t}_{r:02d}_{c:02d}\\\' for t in times for r in range(70) for c in range(35)]\\n    assert len(ids)==646800 and len(template)==646800 and template.id.is_unique and set(ids)==set(template.id)\\n    d=pd.DataFrame(p.reshape(-1,4),columns=[\\\'RH\\\',\\\'T\\\',\\\'U\\\',\\\'V\\\'],index=ids).loc[template.id].reset_index(drop=True); d.insert(0,\\\'id\\\',template.id.to_numpy())\\n    assert np.isfinite(d.iloc[:,1:].to_numpy()).all() and d.RH.between(0,100).all(); tmp=dest.with_suffix(\\\'.tmp\\\'); d.to_csv(tmp,index=False); tmp.replace(dest)\\n    return hashlib.sha256(dest.read_bytes()).hexdigest()\\n\\ndef main(root,out,smoke=False,anchor_only=False):\\n    global LOG\\n    start=time.perf_counter(); out=Path(out); out.mkdir(parents=True,exist_ok=True); LOG=out/\\\'training.log\\\'; LOG.write_text(\\\'\\\')\\n    # Avoid mistaking an older successful run for a new one.\\n    for name in (\\\'submission_atlas_v3_1.csv\\\',\\\'report.json\\\'):\\n        old=out/name\\n        if old.exists(): old.rename(out/(name+\\\'.previous\\\'))\\n    torch.set_num_threads(4 if smoke or anchor_only else 2); train,test,s,template=load_all(root); Y=train[\\\'Y\\\']; T=train[\\\'times\\\'].astype(str); TT=test[\\\'times\\\'].astype(str)\\n    assert s[\\\'vars\\\'].tolist()==[\\\'RH\\\',\\\'T\\\',\\\'U\\\',\\\'V\\\'] and np.all(T[1:]>T[:-1]) and np.all(TT[1:]>TT[:-1]) and T[-1]<TT[0]\\n    assert np.isfinite(Y).all() and np.isfinite(train[\\\'X\\\']).all() and np.isfinite(test[\\\'X\\\']).all()\\n    X,t,h=prepare(train[\\\'X\\\'],T,s); XT,tT,hT=prepare(test[\\\'X\\\'],TT,s)\\n    devices=[torch.device(\\\'cpu\\\')] if smoke or anchor_only else [torch.device(f\\\'cuda:{i}\\\') for i in range(min(2,torch.cuda.device_count()))]\\n    if not devices: raise RuntimeError(\\\'Seleccionar GPU T4 x2\\\')\\n    epochs=1 if smoke else 48; log(\\\'Atlas v3.1\\\',devices,\\\'smoke\\\',smoke,\\\'anchor_only\\\',anchor_only,\\\'epochs\\\',epochs)\\n    rows=[]; models=[\\\'RawResidual\\\',\\\'PhysicsResidual\\\',\\\'PhysicalResidual\\\']; chosen=None; learned=[]\\n    for i,(name,cut,cal,end) in enumerate(CUTS[:1] if smoke else CUTS):\\n        tr=np.where(T<cut)[0]; ca=np.where((T>=cut)&(T<cal)&(h%3==0))[0]; ev=np.where((T>=cal)&(T<end)&(h%3==0))[0]\\n        assert T[tr[-1]]<T[ca[0]] and T[ca[-1]]<T[ev[0]]\\n        if smoke: tr=tr[::13]\\n        both=np.r_[ca,ev]; at,ae=anchor(X,t,h,Y,tr,both,s); a=ae[len(ca):]; score,rm=metric(a,Y[ev]); rows.append(dict(fold=name,model=\\\'Anchor\\\',skill=score,reference=BASE[name],**dict(zip([\\\'RMSE_RH\\\',\\\'RMSE_T\\\',\\\'RMSE_U\\\',\\\'RMSE_V\\\'],rm.tolist())))); log(rows[-1])\\n        if not anchor_only:\\n            for kind in (models if i<2 else [chosen]):\\n                pred=ensemble(X,t,Y,tr,both,at,ae,s,devices,epochs,kind,smoke)\\n                w=weights(ae[:len(ca)],pred[:len(ca)],Y[ca]) if i<2 else np.mean([w for k,w in learned if k==kind],0); q=blend(a,pred[len(ca):],w); score,rm=metric(q,Y[ev]); rows.append(dict(fold=name,model=kind,skill=score,reference=BASE[name],weights=w.tolist(),**dict(zip([\\\'RMSE_RH\\\',\\\'RMSE_T\\\',\\\'RMSE_U\\\',\\\'RMSE_V\\\'],rm.tolist())))); log(rows[-1]); np.savez_compressed(out/f\\\'{name.replace(" ","_")}_{kind}.npz\\\',times=T[ev],pred=q,anchor=a,neural=pred[len(ca):])\\n                if i<2: learned.append((kind,w))\\n            if i==1:\\n                chosen=max(models,key=lambda k:np.mean([r[\\\'skill\\\'] for r in rows if r[\\\'model\\\']==k])); log(\\\'FROZEN BEFORE AUDIT\\\',chosen)\\n        pd.DataFrame(rows).to_csv(out/\\\'validation_metrics.csv\\\',index=False)\\n    report=dict(model=\\\'Atlas v3.1\\\',smoke=smoke,anchor_only=anchor_only,results=rows,recommend_submit=False)\\n    if not anchor_only:\\n        if smoke: chosen=models[0]\\n        gains={r[\\\'fold\\\']:r[\\\'skill\\\']-r[\\\'reference\\\'] for r in rows if r[\\\'model\\\']==chosen}; gate=not smoke and len(gains)==4 and min(gains[\\\'Mayo\\\'],gains[\\\'Junio I\\\'])>0 and np.mean([gains[\\\'Mayo\\\'],gains[\\\'Junio I\\\']])>=.003 and gains[\\\'Junio II\\\']>=.003 and gains[\\\'Abril\\\']>=-.002\\n        report.update(chosen=chosen,gains=gains,recommend_submit=bool(gate)); log(\\\'VALIDATION_GATE\\\',gate,\\\'gains\\\',gains)\\n        if gate or smoke:\\n            Xall=np.concatenate([X,XT]); tall=np.concatenate([t,tT]); hall=np.r_[h,hT]; tr=np.arange(len(T)); ev=len(T)+np.where(hT%3==0)[0]\\n            if smoke: tr=tr[::13]\\n            at,ae=anchor(Xall,tall,hall,Y,tr,ev,s); pred=ensemble(Xall,tall,Y,tr,ev,at,ae,s,devices,epochs,chosen,smoke,out); w=np.mean([w for k,w in learned if k==chosen],0); p=blend(ae,pred,w)\\n            dest=out/(\\\'SMOKE_NO_SUBIR.csv\\\' if smoke else \\\'submission_atlas_v3_1.csv\\\'); sha=save_csv(template,TT[hT%3==0],p,dest); report.update(weights=w.tolist(),sha256=sha,rows=len(template),csv=str(dest)); log(dest,sha)\\n        else: log(\\\'No CSV competitivo. Conservar Ultralisk v2.7.\\\')\\n    report[\\\'seconds\\\']=time.perf_counter()-start; (out/\\\'report.json\\\').write_text(json.dumps(report,indent=2)); log(\\\'RECOMMEND_SUBMIT=\\\'+(\\\'YES\\\' if report[\\\'recommend_submit\\\'] else \\\'NO\\\'))\\n\\n\'\nULTRA_SOURCE=\'# Ultralisk-v2.4 | ShadowBytes | ApuCast | Kaggle T4 x2\\nimport os, time, json, copy, hashlib, threading, platform\\nfrom pathlib import Path\\nfrom concurrent.futures import ThreadPoolExecutor\\nimport numpy as np\\nimport pandas as pd\\nimport torch\\nfrom torch import nn\\nfrom torch.nn import functional as F\\nfrom IPython.display import display, FileLink\\nSMOKE=os.environ.get(\\\'ULTRALISK_SMOKE\\\')==\\\'1\\\'  # prueba tecnica, NO subir\\nOUT=Path(os.environ.get(\\\'ULTRALISK_OUT\\\',\\\'/kaggle/working/Ultralisk-v2.4\\\')); OUT.mkdir(parents=True,exist_ok=True)\\nLOG=OUT/\\\'training.log\\\'; LOG.write_text(\\\'\\\'); LOCK=threading.Lock(); START=time.perf_counter()\\nBATCH=32; EPOCHS_REF=25; EPOCHS_NEW=36\\nSEEDS={\\\'CNN\\\':(0,1,2),\\\'ResNet\\\':(0,1,2),\\\'Mutalisk\\\':(42,123),\\\'Viper\\\':(42,123)}\\nCUTS=[(\\\'2024050900\\\',\\\'2024051600\\\',\\\'2024060100\\\'),(\\\'2024052500\\\',\\\'2024060100\\\',\\\'2024061600\\\'),(\\\'2024060900\\\',\\\'2024061600\\\',\\\'2024070100\\\')]\\nVARS=[\\\'RH\\\',\\\'T\\\',\\\'U\\\',\\\'V\\\']; NORM=np.array([25.360,2.0652,2.6776,2.0876])\\n# Valores historicos de versions/v3/backtest.json (solo referencia, no etiquetas).\\nHISTORICAL={1:.42292221906027727,2:.4136267261457105,3:.42723285641743414}\\nMIN_DEV_GAIN=.003; MIN_AUDIT_GAIN=.002\\ndef log(msg):\\n    line=f\\\'[{time.perf_counter()-START:8.1f}s] {msg}\\\'\\n    with LOCK:\\n        print(line,flush=True)\\n        with LOG.open(\\\'a\\\') as f: f.write(line+\\\'\\\\n\\\')\\nif not SMOKE and (torch.cuda.device_count()!=2 or any(\\\'T4\\\' not in torch.cuda.get_device_name(i) for i in range(2))): raise RuntimeError(\\\'Selecciona Accelerator > GPU T4 x2\\\')\\nDEVICES=[torch.device(\\\'cpu\\\')] if SMOKE else [torch.device(\\\'cuda:0\\\'),torch.device(\\\'cuda:1\\\')]\\ntorch.set_num_threads(2); torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True\\ncpu_name=platform.processor()\\nif Path(\\\'/proc/cpuinfo\\\').is_file():\\n    cpu_name=next((l.split(\\\':\\\',1)[1].strip() for l in Path(\\\'/proc/cpuinfo\\\').read_text().splitlines() if l.startswith(\\\'model name\\\')),cpu_name)\\nlog(f\\\'torch={torch.__version__} CPU={cpu_name} cores={os.cpu_count()} torch_threads={torch.get_num_threads()} smoke={SMOKE}\\\')\\nfor d in DEVICES:\\n    if d.type==\\\'cuda\\\': log(f\\\'{d} {torch.cuda.get_device_name(d)} VRAM={torch.cuda.get_device_properties(d).total_memory/2**30:.2f} GiB\\\')\\n# Datos oficiales\\nroot=Path(os.environ.get(\\\'ULTRALISK_DATA\\\',\\\'/kaggle/input\\\')); required=(\\\'train.npz\\\',\\\'test.npz\\\',\\\'static.npz\\\',\\\'sample_submission.csv\\\')\\nfolders=[p.parent for p in root.rglob(\\\'train.npz\\\') if all((p.parent/k).is_file() for k in required)]\\nofficial=[p for p in folders if \\\'hackathon-ia-pucp-2026\\\' in str(p)]\\nif len(official)==1: DATA=official[0]\\nelif len(folders)==1: DATA=folders[0]\\nelse: raise RuntimeError(f\\\'Dataset oficial no identificado: {folders}\\\')\\nwith np.load(DATA/\\\'train.npz\\\') as f: X,Y,TT=f[\\\'X\\\'].astype(\\\'f4\\\'),f[\\\'Y\\\'].astype(\\\'f4\\\'),f[\\\'times\\\'].astype(str)\\nwith np.load(DATA/\\\'test.npz\\\') as f: XT,TE=f[\\\'X\\\'].astype(\\\'f4\\\'),f[\\\'times\\\'].astype(str)\\nwith np.load(DATA/\\\'static.npz\\\') as f:\\n    DEM=f[\\\'dem\\\'].astype(\\\'f4\\\'); glat,glon,wlat,wlon=[f[k] for k in (\\\'gfs_lats\\\',\\\'gfs_lons\\\',\\\'wrf_lats\\\',\\\'wrf_lons\\\')]\\nTEMPLATE=pd.read_csv(DATA/\\\'sample_submission.csv\\\',dtype={\\\'id\\\':str})\\nassert X.shape==(2935,17,13,4) and Y.shape==(2935,70,35,4) and XT.shape==(792,17,13,4) and DEM.shape==(70,35)\\nassert TEMPLATE.shape==(646800,5) and TEMPLATE.columns.tolist()==[\\\'id\\\',*VARS] and TEMPLATE.id.is_unique\\nassert all(np.isfinite(a).all() for a in (X,Y,XT,DEM)) and max(TT)<min(TE)\\nparts=TEMPLATE.id.str.split(\\\'_\\\',expand=True); ids_t=parts[0].map({s:i for i,s in enumerate(TE)}); assert ids_t.notna().all()\\nti=ids_t.to_numpy(dtype=int); ri=parts[1].astype(int).to_numpy(); ci=parts[2].astype(int).to_numpy()\\nassert (ri>=0).all() and (ri<70).all() and (ci>=0).all() and (ci<35).all()\\nassert len(np.unique(ti))==264 and np.all(np.bincount(ti)[np.unique(ti)]==2450)\\nHOURS=np.unique([int(s[-2:]) for s in parts[0].unique()]); H=np.array([int(s[-2:]) for s in TT]); HT=np.array([int(s[-2:]) for s in TE])\\nlog(f\\\'Data={DATA} train={X.shape} test={XT.shape} horas={HOURS.tolist()}\\\')\\ndef windows(x,times):\\n    dates=pd.to_datetime(times,format=\\\'%Y%m%d%H\\\'); lookup={t:i for i,t in enumerate(dates)}\\n    return x[np.array([[lookup.get(t-pd.Timedelta(hours=k),i) for k in (2,1,0)] for i,t in enumerate(dates)])]\\nW,WT=windows(X,TT),windows(XT,TE)\\nlat,lon=np.meshgrid(wlat,wlon,indexing=\\\'ij\\\'); grid=np.stack([(lon-glon[0])/(glon[-1]-glon[0])*2-1,(lat-glat[0])/(glat[-1]-glat[0])*2-1],-1)[None].astype(\\\'f4\\\')\\ndef z(a): return ((a-a.mean())/max(float(a.std()),1e-6)).astype(\\\'f4\\\')\\ndef smooth(a,k):\\n    p=k//2; return F.avg_pool2d(F.pad(torch.from_numpy(a[None,None]),(p,p,p,p),mode=\\\'replicate\\\'),k,stride=1)[0,0].numpy()\\ndy,dx=np.gradient(DEM)\\ngeo=np.stack([z(DEM),z(dx),z(dy),z(np.hypot(dx,dy)),z(DEM-smooth(DEM,3)),z(DEM-smooth(DEM,7)),z(DEM-smooth(DEM,15)),z(lat),z(lon),z(np.gradient(dy,axis=0)+np.gradient(dx,axis=1))])[None].astype(\\\'f4\\\')\\ndef stats(ix):\\n    yy,xx,hh=Y[ix],X[ix],H[ix]\\n    def climate(a): return np.stack([a[hh==h].mean(0) if np.any(hh==h) else a.mean(0) for h in range(24)])\\n    cl=climate(yy)\\n    return dict(mu=xx.mean((0,1,2)),sd=np.maximum(xx.std((0,1,2)),1e-6),clim=cl,gfs_clim=climate(xx),scale=np.maximum((yy-cl[hh]).std((0,1,2)),1e-6))\\ndef score(p,y):\\n    rm=np.sqrt(np.mean(np.square(p.astype(\\\'f8\\\')-y),axis=(0,1,2))); return float(1-(rm/NORM).mean()),rm\\ndef clip(p):\\n    p=p.astype(\\\'f4\\\',copy=False); p[...,0]=p[...,0].clip(0,100); assert np.isfinite(p).all(); return p\\ndef tensor(a,d): return torch.as_tensor(a,dtype=torch.float32,device=d)\\ndef sync(d):\\n    if d.type==\\\'cuda\\\': torch.cuda.synchronize(d)\\n# Arquitecturas\\nclass Residual(nn.Module):\\n    def __init__(self,a,b,dilation=1):\\n        super().__init__(); self.c1=nn.Conv2d(a,b,3,padding=dilation,dilation=dilation); self.c2=nn.Conv2d(b,b,3,padding=1); self.skip=nn.Conv2d(a,b,1) if a!=b else nn.Identity()\\n    def forward(self,x): return F.gelu(self.c2(F.gelu(self.c1(x)))+self.skip(x))\\ndef inputs(x,h):\\n    n=len(x); a=h*(2*np.pi/24); clock=torch.stack([a.sin(),a.cos()],1)[:,:,None,None].expand(-1,-1,17,13)\\n    return x.permute(0,1,4,2,3).reshape(n,12,17,13),clock\\nclass Reference(nn.Module):\\n    def __init__(self,kind):\\n        super().__init__(); w=64\\n        self.enc=nn.Sequential(nn.Conv2d(14,w,3,padding=1),nn.GELU(),nn.Conv2d(w,w,3,padding=1),nn.GELU())\\n        self.static=nn.Parameter(torch.zeros(1,8,70,35))\\n        self.dec=nn.Sequential(nn.Conv2d(73,w,3,padding=1),nn.GELU(),nn.Conv2d(w,w,3,padding=1),nn.GELU(),nn.Conv2d(w,4,1))\\n        if kind==\\\'ResNet\\\': self.enc=Residual(14,w); self.dec=nn.Sequential(Residual(73,w),nn.Conv2d(w,4,1))\\n        self.register_buffer(\\\'grid\\\',torch.from_numpy(grid)); self.register_buffer(\\\'dem\\\',torch.from_numpy(z(DEM)[None,None]))\\n    def forward(self,x,h):\\n        n=len(x); fields,clock=inputs(x,h); v=self.enc(torch.cat([fields,clock],1))\\n        with torch.autocast(device_type=x.device.type,enabled=False): v=F.grid_sample(v.float(),self.grid.expand(n,-1,-1,-1),align_corners=True)\\n        return self.dec(torch.cat([v,self.dem.expand(n,-1,-1,-1),self.static.expand(n,-1,-1,-1)],1)).permute(0,2,3,1)\\nclass GeoWind(nn.Module):\\n    def __init__(self,kind,st):\\n        super().__init__(); self.kind=kind; w=48\\n        self.register_buffer(\\\'grid\\\',torch.from_numpy(grid)); self.register_buffer(\\\'geo\\\',torch.from_numpy(geo)); self.register_buffer(\\\'gcl\\\',torch.from_numpy(((st[\\\'gfs_clim\\\']-st[\\\'mu\\\'])/st[\\\'sd\\\']).astype(\\\'f4\\\')))\\n        self.enc=nn.Sequential(Residual(22,w),Residual(w,w,2),Residual(w,w)); self.map=nn.Parameter(torch.zeros(1,8,70,35)); self.fuse=Residual(w+26,w)\\n        if kind==\\\'Mutalisk\\\': self.down=nn.Sequential(Residual(w,64),Residual(64,64,2)); self.up=Residual(w+64,w)\\n        else: self.wind=nn.Sequential(Residual(w,w,2),Residual(w,w,3))\\n        self.thermal=nn.Sequential(Residual(w,32),nn.Conv2d(32,2,1)); self.velocity=nn.Sequential(Residual(w,32),nn.Conv2d(32,2,1))\\n    def interp(self,x):\\n        with torch.autocast(device_type=x.device.type,enabled=False): return F.grid_sample(x.float(),self.grid.expand(len(x),-1,-1,-1),align_corners=True,padding_mode=\\\'border\\\')\\n    def forward(self,x,h):\\n        n=len(x); fields,clock=inputs(x,h); current=x[:,-1]; anomaly=current-self.gcl[h.long()]; trend=current-x[:,0]\\n        av=anomaly.permute(0,3,1,2); cv=current.permute(0,3,1,2); v=self.enc(torch.cat([fields,av,trend.permute(0,3,1,2),clock],1))\\n        fine=self.fuse(torch.cat([self.interp(v),self.interp(torch.cat([cv,av],1)),self.geo.expand(n,-1,-1,-1),self.map.expand(n,-1,-1,-1)],1))\\n        if self.kind==\\\'Mutalisk\\\':\\n            low=self.down(F.avg_pool2d(fine,2)); fine=self.up(torch.cat([fine,F.interpolate(low,size=fine.shape[-2:],mode=\\\'bilinear\\\',align_corners=False)],1)); wind=fine\\n        else: wind=self.wind(fine)\\n        return torch.cat([self.thermal(fine),self.velocity(wind)],1).permute(0,2,3,1)\\ndef make(kind,st): return Reference(kind) if kind in (\\\'CNN\\\',\\\'ResNet\\\') else GeoWind(kind,st)\\n# Benchmark de inferencia, mismo modelo y lote para todos los dispositivos.\\ndef benchmark():\\n    st=stats(np.arange(128)); torch.manual_seed(0); base=Reference(\\\'ResNet\\\').eval(); results=[]\\n    x=(W[:8]-st[\\\'mu\\\'])/st[\\\'sd\\\']; h=H[:8]\\n    for d,amp in [(torch.device(\\\'cpu\\\'),False)]+[(d,a) for d in DEVICES if d.type==\\\'cuda\\\' for a in (False,True)]:\\n        m=copy.deepcopy(base).to(d,memory_format=torch.channels_last); xb,hb=tensor(x,d),tensor(h,d)\\n        with torch.inference_mode():\\n            for _ in range(3):\\n                with torch.autocast(device_type=d.type,dtype=torch.float16,enabled=amp): m(xb,hb)\\n            sync(d); t=time.perf_counter()\\n            for _ in range(8):\\n                with torch.autocast(device_type=d.type,dtype=torch.float16,enabled=amp): m(xb,hb)\\n            sync(d); sec=time.perf_counter()-t\\n        r=dict(device=str(d),amp=amp,batch=8,ms_per_batch=sec/8*1000,samples_per_second=64/sec); results.append(r); log(f\\\'BENCHMARK {r}\\\')\\n    if len(DEVICES)==2:\\n        barrier=threading.Barrier(2)\\n        def both(d):\\n            m=copy.deepcopy(base).to(d,memory_format=torch.channels_last); xb,hb=tensor(x,d),tensor(h,d)\\n            with torch.inference_mode(), torch.autocast(device_type=\\\'cuda\\\',dtype=torch.float16):\\n                for _ in range(3): m(xb,hb)\\n                sync(d); barrier.wait(timeout=60); t=time.perf_counter()\\n                for _ in range(8): m(xb,hb)\\n                sync(d); return time.perf_counter()-t\\n        with ThreadPoolExecutor(max_workers=2) as pool:\\n            sec=max(pool.map(both,DEVICES))\\n        r=dict(device=\\\'T4x2_parallel\\\',amp=True,batch=16,ms_per_batch=sec/8*1000,samples_per_second=128/sec)\\n        results.append(r); log(f\\\'BENCHMARK {r}\\\')\\n    return results\\nBENCH=benchmark() if os.environ.get(\\\'ULTRALISK_BENCH\\\',\\\'0\\\')==\\\'1\\\' else []\\nlog(\\\'Benchmark opcional; iniciando experimentos v2.4\\\')\\n@torch.inference_mode()\\ndef infer(model,x,h,d,amp):\\n    model.eval(); out=[]\\n    for i in range(0,len(x),BATCH):\\n        with torch.autocast(device_type=d.type,dtype=torch.float16,enabled=amp): p=model(x[i:i+BATCH],h[i:i+BATCH])\\n        out.append(p.float().cpu().numpy())\\n    return np.concatenate(out)\\ndef fit_neural(job,d,st,tr,we,he,tag):\\n    kind,seed,model=job; ref=kind in (\\\'CNN\\\',\\\'ResNet\\\'); epochs=1 if SMOKE else (EPOCHS_REF if ref else EPOCHS_NEW)\\n    amp=not ref and d.type==\\\'cuda\\\'; lr=.002 if ref else .0012; rng=np.random.RandomState(seed)\\n    model=model.to(d,memory_format=torch.channels_last); ema=None if ref else copy.deepcopy(model).eval().requires_grad_(False)\\n    x=tensor((W[tr]-st[\\\'mu\\\'])/st[\\\'sd\\\'],d); h=tensor(H[tr],d); y=tensor((Y[tr]-st[\\\'clim\\\'][H[tr]])/st[\\\'scale\\\'],d)\\n    xe=tensor((we-st[\\\'mu\\\'])/st[\\\'sd\\\'],d); he_gpu=tensor(he,d)\\n    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4 if ref else 5e-4)\\n    steps=(len(tr)+BATCH-1)//BATCH; sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=lr,total_steps=steps*epochs)\\n    scaler=torch.amp.GradScaler(\\\'cuda\\\',enabled=amp); ratio=tensor(st[\\\'scale\\\']/NORM,d)\\n    order=np.arange(len(tr)); started=time.perf_counter(); hist=[]\\n    if d.type==\\\'cuda\\\': torch.cuda.reset_peak_memory_stats(d)\\n    log(f\\\'{tag} {kind}:{seed} {d} START params={sum(p.numel() for p in model.parameters()):,} epochs={epochs} amp={amp}\\\')\\n    for ep in range(1,epochs+1):\\n        model.train(); rng.shuffle(order); accum=torch.zeros((),device=d); t0=time.perf_counter()\\n        for k in range(0,len(order),BATCH):\\n            ix=torch.as_tensor(order[k:k+BATCH],device=d,dtype=torch.long); opt.zero_grad(set_to_none=True)\\n            with torch.autocast(device_type=d.type,dtype=torch.float16,enabled=amp): p=model(x[ix],h[ix])\\n            mse=(p.float()-y[ix]).square(); loss=mse.mean() if ref else .5*mse.mean()+.5*((mse.mean((0,1,2))+1e-8).sqrt()*ratio).mean()\\n            scaler.scale(loss).backward()\\n            if not ref: scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(),1.)\\n            old=scaler.get_scale(); scaler.step(opt); scaler.update()\\n            if scaler.get_scale()>=old:\\n                sched.step()\\n                if ema is not None:\\n                    with torch.no_grad():\\n                        for a,b in zip(ema.parameters(),model.parameters()): a.lerp_(b,.015)\\n            accum+=loss.detach()*len(ix)\\n        sync(d); sec=time.perf_counter()-t0; value=float(accum.item()/len(tr)); assert np.isfinite(value)\\n        mem=torch.cuda.max_memory_allocated(d)/2**30 if d.type==\\\'cuda\\\' else None\\n        hist.append(dict(epoch=ep,loss=value,seconds=sec,samples_per_second=len(tr)/sec,peak_gib=mem,lr=opt.param_groups[0][\\\'lr\\\']))\\n        log(f\\\'{tag} {kind}:{seed} {d} ep={ep:02d}/{epochs} loss={value:.5f} lr={opt.param_groups[0]["lr"]:.6f} {sec:.2f}s {len(tr)/sec:.1f} samples/s VRAM={mem}\\\')\\n    chosen=model if ref else ema; p=clip(infer(chosen,xe,he_gpu,d,amp)*st[\\\'scale\\\']+st[\\\'clim\\\'][he]); elapsed=time.perf_counter()-started\\n    pd.DataFrame(hist).to_csv(OUT/f\\\'{tag}_{kind}_{seed}_history.csv\\\',index=False)\\n    if tag==\\\'final\\\': torch.save({\\\'kind\\\':kind,\\\'seed\\\':seed,\\\'epochs\\\':epochs,\\\'state_dict\\\':{k:v.detach().cpu() for k,v in chosen.state_dict().items()}},OUT/f\\\'{kind}_{seed}.pt\\\')\\n    log(f\\\'{tag} {kind}:{seed} DONE {elapsed:.1f}s\\\'); model.to(\\\'cpu\\\')\\n    return kind,p,dict(kind=kind,seed=seed,device=str(d),seconds=elapsed,epochs=epochs,peak_gib=mem)\\n# Kernel Ridge no lineal: anomalias del GFS, tendencia y promedio temporal.\\ndef kernel_features(w,h,st):\\n    a=(w-st[\\\'gfs_clim\\\'][(h[:,None]-np.array([2,1,0]))%24])/st[\\\'sd\\\']; a=a.reshape(len(w),3,-1)\\n    return np.concatenate([a[:,2],a.mean(1),.5*(a[:,2]-a[:,0])],axis=1).astype(\\\'f4\\\')\\ndef kernel(a,b):\\n    dot=(a@b.T)/a.shape[1]; dist=(a.square().mean(1)[:,None]+b.square().mean(1)[None,:]-2*dot).clamp_min(0)\\n    return .5*dot+.25*dot.square()+.25*torch.exp(-dist/2)\\ndef fit_kernel(d,st,tr,we,he,tag):\\n    t=time.perf_counter(); a=kernel_features(W[tr],H[tr],st); b=kernel_features(we,he,st)\\n    mu=a.mean(0); sd=np.maximum(a.std(0),.15); a=(a-mu)/sd; b=(b-mu)/sd\\n    ag,bg=tensor(a,d),tensor(b,d); K=kernel(ag,ag); K=(K+K.T)*.5; K.diagonal().add_(.05)\\n    L=torch.linalg.cholesky(K); yy=tensor(((Y[tr]-st[\\\'clim\\\'][H[tr]])/st[\\\'scale\\\']).reshape(len(tr),-1),d)\\n    dual=torch.cholesky_solve(yy,L); p=(kernel(bg,ag)@dual).cpu().numpy().reshape(len(we),70,35,4)\\n    if tag==\\\'final\\\': np.savez_compressed(OUT/\\\'kernel_model.npz\\\',features=a,mu=mu,sd=sd,dual=dual.cpu().numpy())\\n    log(f\\\'{tag} Kernel DONE {time.perf_counter()-t:.2f}s alpha=.05\\\'); return clip(p*st[\\\'scale\\\']+st[\\\'clim\\\'][he])\\nALL_RUNS=[]\\ndef run_stage(tr,we,he,tag,families):\\n    st=stats(tr); jobs=[]\\n    for kind in families:\\n        if kind==\\\'Kernel\\\': continue\\n        for seed in (SEEDS[kind][:1] if SMOKE else SEEDS[kind]):\\n            torch.manual_seed(seed); jobs.append((kind,seed,make(kind,st)))\\n    def worker(group,d): return [fit_neural(j,d,st,tr,we,he,tag) for j in group]\\n    results=[]\\n    if len(DEVICES)==2:\\n        with ThreadPoolExecutor(max_workers=2) as pool:\\n            futures=[pool.submit(worker,jobs[i::2],d) for i,d in enumerate(DEVICES)]\\n            for f in futures: results.extend(f.result())\\n    else: results=worker(jobs,DEVICES[0])\\n    ALL_RUNS.extend([dict(tag=tag,**r) for _,_,r in results]); results.sort(key=lambda x:x[2][\\\'seed\\\'])\\n    pred={k:np.mean([p for kind,p,_ in results if kind==k],axis=0,dtype=np.float32) for k in families if k!=\\\'Kernel\\\'}\\n    if \\\'Kernel\\\' in families: pred[\\\'Kernel\\\']=fit_kernel(DEVICES[0],st,tr,we,he,tag)\\n    if \\\'Kernel\\\' in families: pred[\\\'Clock\\\']=fit_clock(DEVICES[0],st,tr,we,he,tag)\\n    if tag==\\\'final\\\': np.savez_compressed(OUT/\\\'preprocessing.npz\\\',**st,grid=grid,geo=geo,normalizers=NORM)\\n    return pred\\n# Calibracion por variable, separada de evaluacion.\\nCANDIDATES={\\\'Reference\\\':{},\\\'Mutalisk\\\':{\\\'Mutalisk\\\':1.},\\\'Viper\\\':{\\\'Viper\\\':1.},\\\'Neural\\\':{\\\'Mutalisk\\\':.5,\\\'Viper\\\':.5},\\\'Kernel\\\':{\\\'Kernel\\\':1.},\\\'Hybrid\\\':{\\\'Mutalisk\\\':.375,\\\'Viper\\\':.375,\\\'Kernel\\\':.25}}\\ndef fit_alpha(a,b,y,prior=.5,shrink=.2,cap=1.):\\n    d=b.astype(\\\'f8\\\')-a; r=y.astype(\\\'f8\\\')-a; den=(d*d).mean((0,1,2)); num=(r*d).mean((0,1,2))\\n    raw=np.divide(num,den,out=np.full(4,prior),where=den>1e-12); return np.clip((1-shrink)*raw+shrink*prior,0,cap).astype(\\\'f4\\\')\\ndef mix(a,b,w): return clip(a*(1-w)+b*w)\\ndef candidate(p,name): return sum(p[k]*np.float32(w) for k,w in CANDIDATES[name].items())\\ndef calibrate(p,y,name):\\n    rw=fit_alpha(p[\\\'CNN\\\'],p[\\\'ResNet\\\'],y); ref=mix(p[\\\'CNN\\\'],p[\\\'ResNet\\\'],rw)\\n    alpha=np.zeros(4,dtype=\\\'f4\\\') if name==\\\'Reference\\\' else fit_alpha(ref,candidate(p,name),y,prior=.25,shrink=.35,cap=.65)\\n    return rw,alpha\\ndef combine(p,name,rw,alpha):\\n    ref=mix(p[\\\'CNN\\\'],p[\\\'ResNet\\\'],rw); return ref if name==\\\'Reference\\\' else mix(ref,candidate(p,name),alpha)\\n\\n\\n# ================================================================\\n# ULTRALISK v2.4 | CHRONOS | extension incremental de v2.3\\n# No usa WRF oculto, datos externos ni horas futuras de GFS.\\n# ================================================================\\nfrom scipy.optimize import minimize\\nPUBLIC_REFERENCE=.49056\\nOLD_FOLD={1:.439850572420165,2:.4396332592736687,3:.4496978946404211}\\nSTRESS=(\\\'2024040900\\\',\\\'2024041600\\\',\\\'2024050100\\\')\\n# Dos variantes conservadoras, probadas individualmente en CPU en folds 1 y 2.\\nCLOCK_SETTINGS=[(.6,90),(1.2,90)]\\nFAMILIES=[\\\'CNN\\\',\\\'ResNet\\\',\\\'Mutalisk\\\',\\\'Viper\\\',\\\'Kernel\\\']\\nKEYS=FAMILIES+[\\\'Clock\\\']\\nMETHODS=[\\\'ClockReplace\\\',\\\'ClockBlend\\\',\\\'Stack\\\']\\nMIN_DEV=.002; MIN_AUDIT=.002; STRESS_TOL=.002; REPRO_TOL=.0015\\n\\ndef fit_clock(d,st,tr,we,he,tag):\\n    t=time.perf_counter()\\n    a=kernel_features(W[tr],H[tr],st); b=kernel_features(we,he,st)\\n    mu=a.mean(0); sd=np.maximum(a.std(0),.15)\\n    a=(a-mu)/sd; b=(b-mu)/sd\\n    ag,bg=tensor(a,d),tensor(b,d)\\n    h,eh=tensor(H[tr],d),tensor(he,d)\\n    def clock(a,b): return .4+.6*(1+torch.cos((a[:,None]-b[None,:])*(np.pi/12)))/2\\n    K=kernel(ag,ag)*clock(h,h); Q=kernel(bg,ag)*clock(eh,h)\\n    K=(K+K.T)*.5\\n    stamps=pd.to_datetime(TT[tr],format=\\\'%Y%m%d%H\\\')\\n    age=tensor(np.asarray((stamps.max()-stamps).total_seconds()/86400,dtype=\\\'f4\\\'),d)\\n    # Edad calculada solo desde fechas del entrenamiento. Un dia mas aproxima\\n    # el comienzo del siguiente periodo de calibracion (igual en todos los folds).\\n    age=age+1.\\n    target=tensor(((Y[tr]-st[\\\'clim\\\'][H[tr]])/st[\\\'scale\\\']).reshape(len(tr),-1),d)\\n    preds=[]\\n    for alpha,days in CLOCK_SETTINGS:\\n        Kreg=K.clone(); Kreg.diagonal().add_(alpha*torch.exp(age/days))\\n        L=torch.linalg.cholesky(Kreg); dual=torch.cholesky_solve(target,L)\\n        p=(Q@dual).cpu().numpy().reshape(len(we),70,35,4)\\n        preds.append(clip(p*st[\\\'scale\\\']+st[\\\'clim\\\'][he]))\\n        if tag==\\\'final\\\':\\n            np.savez_compressed(OUT/f\\\'clock_alpha_{alpha}.npz\\\',features=a,mu=mu,sd=sd,dual=dual.cpu().numpy(),hours=H[tr],alpha=alpha,days=days)\\n        del Kreg,L,dual,p\\n        log(f\\\'{tag} Clock alpha={alpha} days={days} done elapsed={time.perf_counter()-t:.2f}s\\\')\\n    return clip(np.mean(preds,axis=0,dtype=np.float32))\\n\\ndef old_weights(p,y):\\n    rw,alpha=calibrate(p,y,\\\'Hybrid\\\')\\n    # Convertir mezcla jerarquica v2.3 en pesos efectivos por variable.\\n    return np.stack([(1-alpha)*(1-rw),(1-alpha)*rw,.375*alpha,.375*alpha,.25*alpha,np.zeros(4,dtype=\\\'f4\\\')])\\n\\ndef weighted(p,w):\\n    return clip(sum(p[k]*w[i].astype(\\\'f4\\\') for i,k in enumerate(KEYS)))\\n\\ndef fit_method(p,y,method):\\n    w=old_weights(p,y)\\n    if method==\\\'Baseline\\\': return w\\n    if method==\\\'ClockReplace\\\':\\n        # Misma arquitectura y calibracion que v2.3, sustituyendo 75% del\\n        # componente kernel por la variante horaria/reciente.\\n        ref=mix(p[\\\'CNN\\\'],p[\\\'ResNet\\\'],fit_alpha(p[\\\'CNN\\\'],p[\\\'ResNet\\\'],y))\\n        fresh=.375*p[\\\'Mutalisk\\\']+.375*p[\\\'Viper\\\']+.0625*p[\\\'Kernel\\\']+.1875*p[\\\'Clock\\\']\\n        alpha=fit_alpha(ref,fresh,y,prior=.25,shrink=.35,cap=.65)\\n        rw=fit_alpha(p[\\\'CNN\\\'],p[\\\'ResNet\\\'],y)\\n        return np.stack([(1-alpha)*(1-rw),(1-alpha)*rw,.375*alpha,.375*alpha,.0625*alpha,.1875*alpha])\\n    if method==\\\'ClockBlend\\\':\\n        a=fit_alpha(weighted(p,w),p[\\\'Clock\\\'],y,prior=.15,shrink=.4,cap=.55)\\n        w=w*(1-a); w[-1]+=a; return w\\n    if method!=\\\'Stack\\\': raise ValueError(method)\\n    # Regresion convexa multisalida. Se ajusta solo en la semana de calibracion.\\n    # Penalizacion en escala de RMSE oficial y limites para evitar pesos extremos.\\n    out=[]\\n    for v in range(4):\\n        prior=w[:,v].astype(\\\'f8\\\')*.85; prior[-1]+=.15\\n        # Residuos normalizados, matrices de solo 6x6.\\n        a=np.stack([((p[k][...,v]-y[...,v])/NORM[v]).reshape(-1) for k in KEYS],axis=0).astype(\\\'f8\\\')\\n        gram=(a@a.T)/a.shape[1]; reg=.004\\n        def fun(z): return float(z@gram@z+reg*np.square(z-prior).sum())\\n        def jac(z): return 2*(gram@z+reg*(z-prior))\\n        bounds=[(max(0,float(t)-.25),min(1,float(t)+.25)) for t in prior]; bounds[-1]=(0,.55)\\n        opt=minimize(fun,prior,jac=jac,bounds=bounds,constraints=[dict(type=\\\'eq\\\',fun=lambda z:z.sum()-1,jac=lambda z:np.ones(6))],method=\\\'SLSQP\\\',options=dict(maxiter=100,ftol=1e-11))\\n        z=opt.x if opt.success and np.isfinite(opt.x).all() and abs(opt.x.sum()-1)<1e-6 else prior\\n        z=np.maximum(z,0); out.append(z/z.sum())\\n    return np.asarray(out,dtype=\\\'f4\\\').T\\n\\n# Cache de predicciones de calibracion y validacion para futuras versiones.\\n# Las etiquetas se obtienen exclusivamente de train.npz.\\ndef regional(d,st,tr,we,he,alpha=1.2,sigma=.65,local=.7,days=90,save_model=False,model_prefix="regional"):\\n    a=kernel_features(W[tr],H[tr],st); b=kernel_features(we,he,st); mu=a.mean(0); sd=np.maximum(a.std(0),.15)\\n    ag,bg=tensor((a-mu)/sd,d),tensor((b-mu)/sd,d)\\n    G=kernel(ag,ag); Q=kernel(bg,ag)\\n    h,eh=tensor(H[tr],d),tensor(he,d)\\n    clock=lambda a,b: .4+.6*(1+torch.cos((a[:,None]-b[None,:])*np.pi/12))/2\\n    C,D=clock(h,h),clock(eh,h)\\n    stamps=pd.to_datetime(TT[tr],format=\\\'%Y%m%d%H\\\'); age=tensor(np.asarray((stamps.max()-stamps).total_seconds()/86400,dtype=\\\'f4\\\')+1,d)\\n    yy=((Y[tr]-st[\\\'clim\\\'][H[tr]])/st[\\\'scale\\\']).reshape(len(tr),-1,4); pred=np.zeros((len(we),2450,4),np.float32); total=np.zeros(2450,np.float32)\\n    gy,gx=np.meshgrid(glat,glon,indexing=\\\'ij\\\'); fy,fx=lat.ravel(),lon.ravel()\\n    if save_model: np.savez(OUT/f"{model_prefix}_preprocessing.npz",**st,features=(a-mu)/sd,feature_mu=mu,feature_sd=sd,train_hours=H[tr],train_times=TT[tr],alpha=alpha,sigma=sigma,local=local,days=days)\\n    anchor=0\\n    for cy in np.linspace(wlat[0],wlat[-1],3):\\n        for cx in np.linspace(wlon[0],wlon[-1],2):\\n            ww=np.exp(-.5*((gy-cy)**2+(gx-cx)**2)/sigma**2); ww=(ww/max(ww.mean(),1e-6))**.5; w=tensor(np.tile(np.repeat(ww.ravel(),4),3),d)\\n            k=((1-local)*G+local*kernel(ag*w,ag*w))*C; q=((1-local)*Q+local*kernel(bg*w,ag*w))*D\\n            k=(k+k.T)*.5; k.diagonal().add_(alpha*torch.exp(age/days)); L=torch.linalg.cholesky(k)\\n            blend=np.maximum(1-np.abs(fy-cy)/((wlat[-1]-wlat[0])/2),0)*np.maximum(1-np.abs(fx-cx)/(wlon[-1]-wlon[0]),0)\\n            ids=np.flatnonzero(blend>1e-6); target=tensor(yy[:,ids].reshape(len(tr),-1),d); dual=torch.cholesky_solve(target,L)\\n            if save_model: np.savez(OUT/f"{model_prefix}_sector_{anchor}.npz",indices=ids,blend=blend[ids],feature_weights=w.cpu().numpy(),dual=dual.cpu().numpy(),center=[cy,cx])\\n            anchor+=1\\n            pred[:,ids]+= (q@dual).cpu().numpy().reshape(len(we),len(ids),4)*blend[ids,None]; total[ids]+=blend[ids]\\n    assert total.min()>.99\\n    p=pred.reshape(len(we),70,35,4)/total.reshape(1,70,35,1)\\n    return clip(p*st[\\\'scale\\\']+st[\\\'clim\\\'][he])\\n\\n\\n\\nfrom scipy.ndimage import gaussian_filter\\nANCHOR="Stack"\\nKEYS27=KEYS+["Regional","Narrow"]\\ndef predict27(p,w):\\n    assert w.shape==(len(KEYS27),4) and np.isfinite(w).all() and (w>=-1e-7).all() and np.allclose(w.sum(0),1,atol=1e-5)\\n    return clip(sum(p[k]*w[i].astype(\\\'f4\\\') for i,k in enumerate(KEYS27)))\\n\\ndef padded(w): return np.vstack([w,np.zeros((2,4),dtype=\\\'f4\\\')])\\ndef add_component(w,j,a):\\n    w=w*(1-a); w[j]+=a; return w\\n\\ndef calibration(p,y,method,h):\\n    w=padded(fit_method(p,y,\\\'Baseline\\\' if method==\\\'V23\\\' else ANCHOR))\\n    if method in (\\\'V23\\\',\\\'Anchor\\\'): return w,None\\n    j=6 if method.startswith(\\\'Regional\\\') else 7\\n    if method.endswith(\\\'Replace\\\'):\\n        a=.75*(w[4]+w[5]); w[4:6]*=.25; w[j]+=a\\n    elif method.endswith(\\\'Fixed\\\'): w=add_component(w,j,np.full(4,.20,dtype=\\\'f4\\\'))\\n    elif method in (\\\'NarrowAdaptive\\\',\\\'RegionalAdaptive\\\',\\\'NarrowSpatial\\\',\\\'NarrowCorrected\\\',\\\'NarrowStack\\\'):\\n        a=fit_alpha(predict27(p,w),p[KEYS27[j]],y,prior=.20,shrink=.50,cap=.50)\\n        w=add_component(w,j,a)\\n    else: raise ValueError(method)\\n    if method==\\\'NarrowStack\\\':\\n        result=[]\\n        for v in range(4):\\n            prior=w[:,v].astype(\\\'f8\\\'); prior=prior*.95; prior[6]+=.05\\n            # Regularizar frente a la mezcla jerarquica original, en unidades oficiales.\\n            e=np.stack([((p[k][...,v]-y[...,v])/NORM[v]).ravel() for k in KEYS27]).astype(\\\'f8\\\'); gram=e@e.T/e.shape[1]; reg=.012\\n            bounds=[(max(0,float(x)-.18),min(1,float(x)+.18)) for x in prior]; bounds[7]=(0,.55)\\n            opt=minimize(lambda z:float(z@gram@z+reg*np.square(z-prior).sum()),prior,jac=lambda z:2*(gram@z+reg*(z-prior)),bounds=bounds,constraints=[{\\\'type\\\':\\\'eq\\\',\\\'fun\\\':lambda z:z.sum()-1,\\\'jac\\\':lambda z:np.ones(len(KEYS27))}],method=\\\'SLSQP\\\',options={\\\'maxiter\\\':100,\\\'ftol\\\':1e-11})\\n            q=opt.x if opt.success and np.isfinite(opt.x).all() and abs(opt.x.sum()-1)<1e-6 else prior; q=np.maximum(q,0); result.append(q/q.sum())\\n        w=np.array(result,dtype=\\\'f4\\\').T\\n    maps=None\\n    if method in (\\\'NarrowSpatial\\\',\\\'NarrowCorrected\\\'):\\n        # Dos alternativas: corregir el ensemble o solamente el kernel regional.\\n        e=y-(p[\\\'Narrow\\\'] if method==\\\'NarrowCorrected\\\' else predict27(p,w)); glob=gaussian_filter(e.mean(0),sigma=(3,3,0)); maps=[]\\n        for hour in range(24):\\n            hw=np.exp(1.5*(np.cos((h-hour)*np.pi/12)-1)); local=np.average(e,axis=0,weights=hw)\\n            maps.append(.35*(.5*glob+.5*gaussian_filter(local,sigma=(3,3,0))))\\n        maps=np.asarray(maps,dtype=\\\'f4\\\')\\n        if method==\\\'NarrowCorrected\\\': maps*=w[7]\\n    return w,maps\\n\\ndef apply27(p,params,h):\\n    w,b=params; pred=predict27(p,w); return clip(pred if b is None else pred+b[h])\\n\\n\'\n# Atlas v3.2 | ShadowBytes | paired retrospective audit, not a Kaggle score.\nimport os, argparse, json, hashlib, time, platform, copy, math, threading\nfrom pathlib import Path\nfrom concurrent.futures import ThreadPoolExecutor\nfor k in (\'OPENBLAS_NUM_THREADS\',\'OMP_NUM_THREADS\',\'MKL_NUM_THREADS\'): os.environ[k]=\'4\'\nos.environ.setdefault(\'CUBLAS_WORKSPACE_CONFIG\', \':4096:8\')\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom scipy.linalg import cho_factor, cho_solve\nfrom scipy.ndimage import gaussian_filter\nfrom sklearn.decomposition import PCA\nfrom torch import nn\nimport sklearn, scipy\nNORM=np.array([25.360,2.0652,2.6776,2.0876],dtype=np.float64)\nVARS=[\'RH\',\'T\',\'U\',\'V\']; CUT=\'2024052100\'; CAL=\'2024052800\'; END=\'2024070100\'\nSEEDS=[42,123]; LOCK=threading.Lock(); BASE_HASH=\'9ce6e7594d0edbf884e56902fd67e1fa02de1a94ba2f6815fc5c8b5115a801e1\'\nA={\'__name__\':\'atlas_legacy\'}; exec(ATLAS_SOURCE,A)\n\ndef sha(p):\n    h=hashlib.sha256()\n    with open(p,\'rb\') as f:\n        for b in iter(lambda:f.read(2**20),b\'\'): h.update(b)\n    return h.hexdigest()\ndef dump(p,x): Path(p).write_text(json.dumps(x,indent=2,allow_nan=False))\ndef readnp(p):\n    with np.load(p,allow_pickle=False) as f:return dict(f)\ndef data_root(root):\n    root=Path(root); req=[\'train.npz\',\'test.npz\',\'static.npz\',\'sample_submission.csv\']\n    candidates=[root] if all((root/n).is_file() for n in req) else [p.parent for p in root.rglob(\'train.npz\') if all((p.parent/n).is_file() for n in req)]\n    if len(candidates)!=1: raise ValueError(f\'Dataset ambiguous: {candidates}\')\n    return candidates[0]\ndef clip(x):\n    x=np.asarray(x,dtype=\'f4\').copy(); x[...,0]=np.clip(x[...,0],0,100)\n    if not np.isfinite(x).all(): raise ValueError(\'Nonfinite predictions\')\n    return x\ndef blend(a,b,w):return clip(a+(b-a)*np.asarray(w,dtype=\'f4\'))\ndef sums(p,y):\n    e=p.astype(\'f8\')-y.astype(\'f8\'); return np.square(e).sum(axis=(0,1,2)),int(np.prod(e.shape[:3]))\ndef skill(s,n):return 1-(np.sqrt(np.asarray(s)/np.asarray(n)[...,None])/NORM).mean(-1)\ndef score(p,y):\n    s,n=sums(p,y);return float(skill(s,n))\ndef daystr(t):return np.asarray(t).astype(str).astype(\'U8\')\ndef history(X,t,h,physics):\n    a=X[:,:15] if physics else X[:,:4]; prev=X[:,15:30] if physics else X[:,15:19]; old=X[:,30:45] if physics else X[:,30:34]\n    k=7 if physics else 4\n    b=(a-prev)*t[:,7,None,None,None]; c=(a-old)*t[:,6,None,None,None]\n    return np.concatenate([a,b[:,:k],c[:,:k]],1).transpose(0,2,3,1).copy()\ndef design(z,g,t,group,physics):\n    r,c=divmod(int(group),13); k=7 if physics else 4\n    cells=sorted(set((int(np.clip(r+dr,0,16)),int(np.clip(c+dc,0,12))) for dr,dc in [(0,0),(-1,0),(1,0),(0,-1),(0,1)]))\n    return np.concatenate([z[:,r,c],*[z[:,rr,cc,:k] for rr,cc in cells if (rr,cc)!=(r,c)],g,t[:,:4],t[:,6:]],1).astype(\'f8\')\ndef fit_anchor(X,t,h,Y,tr,s,physics=True):\n    # No held-out X/Y used to fit any statistics, EOF or coefficient.\n    z=history(X[tr],t[tr],h[tr],physics); hh=h[tr]; tt=t[tr]; n=len(tr)\n    cl=A[\'climate\'](z,hh,np.arange(n)); z-=cl[hh]; sd=np.maximum(z.std((0,1,2)),1e-4); z=np.clip(z/sd,-12,12)\n    flat=z.reshape(n,-1); pca=PCA(n_components=min(24,n-1),svd_solver=\'randomized\',random_state=17).fit(flat)\n    g=pca.transform(flat); gs=np.maximum(g.std(0),1e-4); g/=gs\n    yc=A[\'climate\'](Y[tr],hh,np.arange(n)); yy=((Y[tr]-yc[hh])/NORM).reshape(n,-1,4)\n    lat,lon=np.meshgrid(s[\'wrf_lats\'],s[\'wrf_lons\'],indexing=\'ij\'); groups=np.abs(lat.ravel()[:,None]-s[\'gfs_lats\']).argmin(1)*13+np.abs(lon.ravel()[:,None]-s[\'gfs_lons\']).argmin(1)\n    st=dict(physics=np.array(int(physics)),cl=cl,sd=sd,pca_mean=pca.mean_,pca_components=pca.components_,pca_sd=gs,yc=yc,groups=groups)\n    for gr in np.unique(groups):\n        d=design(z,g,tt,gr,physics); mu=d.mean(0); ds=np.maximum(d.std(0),.05); d=np.column_stack([np.ones(n),(d-mu)/ds]); pix=np.flatnonzero(groups==gr)\n        gram=d.T@d; reg=np.full(len(gram),.08*n); reg[0]=1e-6; gram.flat[::len(gram)+1]+=reg\n        beta=cho_solve(cho_factor(gram),d.T@yy[:,pix].reshape(n,-1))\n        for key,v in [(\'mu\',mu),(\'sd\',ds),(\'beta\',beta)]:st[f\'{key}_{gr}\']=v\n    return st\ndef anchor_predict(st,X,t,h):\n    physics=bool(st[\'physics\']); z=history(X,t,h,physics); z=np.clip((z-st[\'cl\'][h])/st[\'sd\'],-12,12)\n    g=((z.reshape(len(z),-1)-st[\'pca_mean\'])@st[\'pca_components\'].T)/st[\'pca_sd\']; groups=st[\'groups\']; pred=np.empty((len(z),2450,4),dtype=\'f4\')\n    for gr in np.unique(groups):\n        d=design(z,g,t,gr,physics); d=np.column_stack([np.ones(len(d)),(d-st[f\'mu_{gr}\'])/st[f\'sd_{gr}\']]); pix=np.flatnonzero(groups==gr); pred[:,pix]=(d@st[f\'beta_{gr}\']).reshape(len(d),len(pix),4)\n    return clip(pred.reshape(-1,70,35,4)*NORM+st[\'yc\'][h])\ndef crossfit(X,t,h,Y,T,tr,s,physics,smoke=False):\n    dates=pd.to_datetime(T,format=\'%Y%m%d%H\'); start=dates[tr[0]].normalize()+pd.Timedelta(days=28); ids=[]; preds=[]; rows=[]\n    for b in pd.date_range(start,dates[tr[-1]].normalize(),freq=\'7D\'):\n        e=b+pd.Timedelta(days=7); fit=tr[dates[tr]<b-pd.Timedelta(hours=24)]; val=tr[(dates[tr]>=b)&(dates[tr]<e)]\n        if len(fit)<30 or not len(val):continue\n        assert dates[fit[-1]]<dates[val[0]]-pd.Timedelta(hours=24)\n        st=fit_anchor(X,t,h,Y,fit,s,physics); q=anchor_predict(st,X[val],t[val],h[val]); ids.append(val);preds.append(q)\n        rows.append(dict(physics=physics,fit_end=str(dates[fit[-1]]),holdout_start=str(b),holdout_end_exclusive=str(e),fit=len(fit),holdout=len(val),gap_hours=24))\n    ix=np.concatenate(ids);assert len(np.unique(ix))==len(ix);return ix,np.concatenate(preds),rows\n\ndef model_new(s,mu,sd,width,physics):\n    geo,grid=A[\'geography\'](s)\n    if not physics: geo=geo.copy();geo[:,1:8]=0 # DEM and coordinates retained; no engineered terrain\n    m=A[\'AtlasResidual\'](geo,grid,s[\'wrf_lats\'],s[\'wrf_lons\'],mu,sd,width=width,physics=physics)\n    m.attn.dropout=0. # fixed across ablations; no shared RNG during parallel training\n    return m\n\ndef train_net(X,t,Y,ix,anchor,s,kind,seed,epochs,device,width,checkpoint):\n    physics=kind!=\'raw\'; wind=kind==\'wind\'; aux=kind==\'aux\'\n    mu=np.tile(X[ix,:15].mean((0,2,3)),3);sd=np.tile(np.maximum(X[ix,:15].std((0,2,3)),1e-4),3)\n    with LOCK:\n        torch.manual_seed(seed); model=model_new(s,mu,sd,width,physics)\n    model=model.to(device); ema=copy.deepcopy(model).eval().requires_grad_(False); rng=np.random.default_rng(seed);batch=32 if width==48 else 16\n    x=torch.as_tensor((X[ix]-mu[None,:,None,None])/sd[None,:,None,None],device=device); tt=torch.as_tensor(t[ix],device=device); aa=torch.as_tensor(anchor,device=device); yy=torch.as_tensor((Y[ix]-anchor)/NORM.astype(\'f4\'),device=device)\n    opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.001);sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=math.ceil(len(ix)/batch)*epochs,eta_min=2e-5); amp=device.type==\'cuda\';scaler=torch.amp.GradScaler(\'cuda\',enabled=amp); updates=0\n    start=time.perf_counter()\n    for ep in range(epochs):\n        model.train(); order=rng.permutation(len(ix))\n        for off in range(0,len(ix),batch):\n            k=torch.as_tensor(order[off:off+batch],device=device);opt.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp):p=model(x[k],tt[k])\n            p=p.float();e=p-yy[k];mse=e.square().mean((0,1,2)); loss_v=.6*(mse+1e-6).sqrt()+.4*mse\n            if wind: loss=.35*loss_v[2]+.65*loss_v[3]+.01*(e[...,2:].diff(dim=1).square().mean()+e[...,2:].diff(dim=2).square().mean())\n            else: loss=loss_v.mean()+(A[\'auxiliary\'](p,yy[k],aa[k]) if aux else 0)\n            if not torch.isfinite(loss):raise FloatingPointError(\'nonfinite loss\')\n            scaler.scale(loss).backward();scaler.unscale_(opt);nn.utils.clip_grad_norm_(model.parameters(),1.);old=scaler.get_scale();scaler.step(opt);scaler.update()\n            if scaler.get_scale()>=old:\n                sched.step();updates+=1\n                with torch.no_grad():torch._foreach_lerp_(list(ema.parameters()),list(model.parameters()),1-min(.995,(1+updates)/(10+updates)))\n        print(kind,seed,ep+1,epochs,round(time.perf_counter()-start,1),flush=True)\n    # Only tensors and primitive metadata; load weights_only=True.\n    torch.save(dict(state=ema.cpu().state_dict(),mu=torch.from_numpy(mu),sd=torch.from_numpy(sd),width=width,kind=kind,seed=seed,epochs=epochs),checkpoint)\n\ndef net_predict(path,X,t,s,anchor,device):\n    ck=torch.load(path,map_location=\'cpu\',weights_only=True);mu=ck[\'mu\'].numpy();sd=ck[\'sd\'].numpy();kind=ck[\'kind\'];model=model_new(s,mu,sd,ck[\'width\'],kind!=\'raw\');model.load_state_dict(ck[\'state\']);model=model.to(device).eval();out=[]\n    with torch.inference_mode():\n        for j in range(0,len(X),32):\n            x=torch.as_tensor((X[j:j+32]-mu[None,:,None,None])/sd[None,:,None,None],device=device);tt=torch.as_tensor(t[j:j+32],device=device)\n            # FP32 inference fixed for both audit and deployment.\n            out.append(model(x,tt).float().cpu().numpy())\n    delta=np.concatenate(out)*NORM\n    if kind==\'wind\':delta[...,:2]=0\n    return clip(anchor+delta)\n\ndef net_family(out,name,kind,X,t,Y,ix,at,s,devices,epochs,width,seeds):\n    paths=[out/f\'{name}_{seed}.pt\' for seed in seeds]\n    def run(j):train_net(X,t,Y,ix,at,s,kind,seeds[j],epochs,devices[j%len(devices)],width,paths[j])\n    with ThreadPoolExecutor(max_workers=len(devices)) as pool:list(pool.map(run,range(len(seeds))))\n    return paths\n\ndef avg_predict(paths,X,t,s,anchor,device):return clip(np.mean([net_predict(p,X,t,s,anchor,device) for p in paths],axis=0,dtype=\'f4\'))\n\ndef bias(p,y,h):\n    e=y-p;g=gaussian_filter(e.mean(0),sigma=(3,3,0));return np.array([.35*(.5*g+.5*gaussian_filter(np.average(e,axis=0,weights=np.exp(1.5*(np.cos((h-i)*np.pi/12)-1))),sigma=(3,3,0))) for i in range(24)],dtype=\'f4\')\ndef optimum(a,b,y,wind=False):\n    d=b.astype(\'f8\')-a.astype(\'f8\');r=y.astype(\'f8\')-a.astype(\'f8\');w=np.clip((d*r).sum((0,1,2))/np.maximum((d*d).sum((0,1,2)),1e-12),0,.5)\n    if wind:w[:2]=0\n    return w\n\ndef chain(a,others,ws):\n    for b,w in zip(others,ws):a=blend(a,b,w)\n    return a\n\ndef fit_chain(a,others,y,shrink,windflags):\n    ws=[]\n    for b,wind in zip(others,windflags):\n        w=shrink*optimum(a,b,y,wind);ws.append(w);a=blend(a,b,w)\n    return ws\n\ndef nested(a,others,y,dates,windflags):\n    # a is already prequential w.r.t. base calibration (no same-day labels).\n    days=np.unique(dates); grid=[0.,.5,.85,1.]; losses=[]\n    for shrink in grid:\n        preds=[];ys=[]\n        for day in days[1:]:\n            tr=dates<day;ev=dates==day;w=fit_chain(a[tr],[b[tr] for b in others],y[tr],shrink,windflags);preds.append(chain(a[ev],[b[ev] for b in others],w));ys.append(y[ev])\n        losses.append(score(np.concatenate(preds),np.concatenate(ys)))\n    best=int(np.argmax(losses));return fit_chain(a,others,y,grid[best],windflags),dict(shrink=grid[best],prequential_scores=dict(zip(map(str,grid),losses)),days=days.tolist())\n\ndef boot(a,b,n,L,B=5000):\n    rng=np.random.default_rng(2026+L);D=len(n);start=rng.integers(0,D-L+1,size=(B,math.ceil(D/L)));idx=(start[...,None]+np.arange(L)).reshape(B,-1)[:,:D];cnt=n[idx].sum(1);good=cnt>0;idx=idx[good];cnt=cnt[good];d=skill(b[idx].sum(1),cnt)-skill(a[idx].sum(1),cnt)\n    return dict(block_calendar_days=L,replicates=len(d),ci95=np.quantile(d,[.025,.975]).tolist(),se=float(d.std(ddof=1)))\n\ndef evaluate(pred,y,T,h,mask,Ytrain,out,base,primary,mode):\n    days=pd.date_range(\'2024-05-28\',\'2024-06-30\').strftime(\'%Y%m%d\').to_numpy();dt=daystr(T); ns=np.array([np.sum(dt==d)*2450 for d in days]); allstats={};rows=[];daily=[];weekly=[];detail=[]\n    def row(name,p,yy,**kw):\n        ss,n=sums(p,yy);rm=np.sqrt(ss/n);return dict(model=name,skill=float(skill(ss,n)),pixels_hours=n,**dict(zip([\'RMSE_\'+v for v in VARS],rm)),**dict(zip([\'normalized_\'+v for v in VARS],rm/NORM)),**kw)\n    quantiles=[np.quantile(Ytrain[...,j],[.05,.95]) for j in range(4)]\n    for name,p in pred.items():\n        rows.append(row(name,p,y));ss=np.array([sums(p[dt==d],y[dt==d])[0] if n else np.zeros(4) for d,n in zip(days,ns)]);allstats[name]=ss\n        for i,(d,n) in enumerate(zip(days,ns)):\n            if n:daily.append(dict(date=d,day_since_cal=i+1,**row(name,p[dt==d],y[dt==d])))\n        for wk in range(5):\n            q=np.isin(dt,days[7*wk:7*(wk+1)])\n            if q.any():weekly.append(row(name,p[q],y[q],week=wk+1))\n        for hr in np.unique(h):detail.append(row(name,p[h==hr],y[h==hr],group=f\'UTC{hr:02}\'))\n        for status in [True,False]:\n            q=(mask[:,:2].sum(1)==2)==status\n            if q.any():detail.append(row(name,p[q],y[q],group=\'history_complete\' if status else \'history_missing\'))\n        # Thresholds fixed by training only. Per-variable tail errors, not a combined score.\n        for j,v in enumerate(VARS):\n            lo,hi=quantiles[j];\n            for tag,q in [(\'low\',y[...,j]<=lo),(\'high\',y[...,j]>=hi)]:\n                if q.any():detail.append(dict(model=name,group=f\'{v}_{tag}_train_quantile\',variable=v,threshold=float(lo if tag==\'low\' else hi),tail_rmse=float(np.sqrt(np.mean((p[...,j][q].astype(\'f8\')-y[...,j][q])**2))),pixels_hours=int(q.sum())))\n    pd.DataFrame(rows).to_csv(out/\'validation_metrics.csv\',index=False);pd.DataFrame(daily).to_csv(out/\'daily_scores.csv\',index=False);pd.DataFrame(weekly).to_csv(out/\'weekly_scores.csv\',index=False);pd.DataFrame(detail).to_csv(out/\'diagnostics.csv\',index=False)\n    a=allstats[base];gates=[]\n    for name,b in allstats.items():\n        if name==base:continue\n        gain=float(skill(b.sum(0),ns.sum())-skill(a.sum(0),ns.sum()));b3=boot(a,b,ns,3);b7=boot(a,b,ns,7);good=ns>0;wins=float(np.mean(skill(b[good],ns[good])>skill(a[good],ns[good])));late=float(skill(b[-14:].sum(0),ns[-14:].sum())-skill(a[-14:].sum(0),ns[-14:].sum())) if ns[-14:].sum() else None\n        threshold=max(.01311,1.96*max(b3[\'se\'],b7[\'se\']));passed=mode==\'full\' and name==primary and gain>=threshold and min(b3[\'ci95\'][0],b7[\'ci95\'][0])>=.003 and wins>=.65 and late is not None and late>=0\n        gates.append(dict(model=name,gain=gain,threshold=threshold,win_fraction=wins,late_gain=late,bootstrap3=b3,bootstrap7=b7,review_eligible=bool(passed)))\n    ablations=[]\n    for first,second in [(\'AtlasAnchorRaw\',\'AtlasAnchor\'),(\'AtlasPhysicsInSample\',\'AtlasPhysicsOOF\'),(\'AtlasRawOOF\',\'AtlasPhysicsOOF\'),(\'AtlasPhysicsOOF\',\'AtlasAuxOOF\'),(\'MixAnchor\',\'MixWind\'),(\'MixPhysics\',\'MixFull\'),(\'MixRaw\',\'MixPhysics\')]:\n        if first in allstats and second in allstats:\n            a0,b0=allstats[first],allstats[second]; ablations.append(dict(reference=first,candidate=second,gain=float(skill(b0.sum(0),ns.sum())-skill(a0.sum(0),ns.sum())),bootstrap3=boot(a0,b0,ns,3),bootstrap7=boot(a0,b0,ns,7)))\n    dump(out/\'paired_ablations.json\',ablations)\n    import matplotlib;matplotlib.use(\'Agg\');import matplotlib.pyplot as plt\n    fig,ax=plt.subplots(figsize=(12,5));df=pd.DataFrame(daily)\n    for name in dict.fromkeys([base,\'AtlasAnchor\',\'AtlasPhysicsOOF\',primary]):\n        z=df[df.model==name];ax.plot(z.day_since_cal,z.skill,marker=\'.\',label=name)\n    ax.set(xlabel=\'Días calendario desde la calibración\',ylabel=\'Skill diario\',title=f\'Atlas v3.2 — {mode}, retrospectivo\');ax.grid(alpha=.2);ax.legend();fig.tight_layout();fig.savefig(out/\'daily_scores.png\',dpi=180);plt.close(fig)\n    return rows,gates\n\ndef fingerprint(data):return {n:sha(data/n) for n in [\'train.npz\',\'test.npz\',\'static.npz\',\'sample_submission.csv\']}\ndef codehash():return sha(Path(__file__))\ndef manifest(out,data_hash,mode,extra=None):\n    files={str(p.relative_to(out)):sha(p) for p in sorted(out.rglob(\'*\')) if p.is_file() and p.name!=\'manifest.json\'}\n    dump(out/\'manifest.json\',dict(model=\'Atlas v3.2\',mode=mode,data=data_hash,code_sha256=codehash(),seeds=SEEDS,cuts=[CUT,CAL,END],versions=dict(python=platform.python_version(),torch=str(torch.__version__),numpy=np.__version__,scipy=scipy.__version__,sklearn=sklearn.__version__,cuda=torch.version.cuda,gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]),tolerance=dict(skill_absolute=.0005,anchor_export_abs_per_value=.0001,checkpoint_roundtrip_abs_per_value=.0001),files=files,extra=extra or {}))\ndef verify_manifest(out,expected_data=None):\n    m=json.loads((out/\'manifest.json\').read_text());assert m[\'code_sha256\']==codehash(),\'Code mismatch\'\n    if expected_data is not None:assert m[\'data\']==expected_data,\'Data mismatch\'\n    for name,h in m[\'files\'].items():\n        if not (out/name).is_file() or sha(out/name)!=h:raise ValueError(f\'Artifact mismatch: {name}\')\n    return m\n\ndef init_ultra(data,out,mode):\n    os.environ.update(ULTRALISK_DATA=str(data),ULTRALISK_OUT=str(out/\'ultra\'),ULTRALISK_SMOKE=\'1\' if mode==\'smoke\' else \'0\');u={\'__name__\':\'ultra_reference\'};exec(ULTRA_SOURCE,u);return u\n\ndef baseline(data,out,mode,tr,ca,ev,T,h,Y,dh,cache_arg):\n    both=np.r_[ca,ev];nc=len(ca);dt=daystr(T[ca]);days=np.unique(dt);base_oof=[];ii=[]\n    if mode==\'proxy\':\n        if not cache_arg:raise ValueError(\'--cache must point to previous proxy validation_predictions.npz\')\n        with np.load(cache_arg,allow_pickle=False) as f:\n            assert np.array_equal(f[\'times\'],T[both]) and np.array_equal(f[\'target\'],Y[both]) and int(f[\'n_cal\'])==nc\n            raw=f[\'NarrowSinSesgo\'];old_anchor=f[\'AtlasAnchor\']\n        def predict(ids,inds):return clip(raw[inds]+bias(raw[ids],Y[ca[ids]],h[ca[ids]])[h[both[inds]]])\n        name=\'NarrowProxy\';extra=dict(source_sha256=sha(cache_arg),source_scope=\'previous proxy; not full Ultralisk\',old_anchor=old_anchor)\n    else:\n        u=init_ultra(data,out,mode);st=u[\'stats\'](tr);fp=hashlib.sha256((json.dumps(dh,sort_keys=True)+hashlib.sha256(ULTRA_SOURCE.encode()).hexdigest()+mode).encode()+tr.tobytes()+both.tobytes()).hexdigest()\n        cache=Path(cache_arg) if cache_arg else out/\'ultra_family_cache.npz\'; p=None\n        if cache.is_file():\n            # Exact split + source + dataset. No historical fold cache can masquerade as this split.\n            with np.load(cache,allow_pickle=False) as f:\n                assert str(f[\'fingerprint\'])==fp and np.array_equal(f[\'times\'],T[both]) and np.array_equal(f[\'target\'],Y[both]);p={k:f[k] for k in u[\'KEYS27\']}\n            assert all(v.shape==Y[both].shape and np.isfinite(v).all() for v in p.values())\n        if p is None:\n            p=u[\'run_stage\'](tr,u[\'W\'][both],h[both],\'long\',u[\'FAMILIES\']);d=u[\'DEVICES\'][0]\n            p[\'Regional\']=u[\'regional\'](d,st,tr,u[\'W\'][both],h[both],sigma=.65,local=.7,days=90)\n            p[\'Narrow\']=u[\'regional\'](d,st,tr,u[\'W\'][both],h[both],sigma=.4,local=.85,days=90)\n            np.savez(cache,fingerprint=fp,times=T[both],target=Y[both],**p)\n        def predict(ids,inds):\n            w,m=u[\'calibration\']({k:v[ids] for k,v in p.items()},Y[ca[ids]],\'NarrowSpatial\',h[ca[ids]])\n            return u[\'apply27\']({k:v[inds] for k,v in p.items()},(w,m),h[both[inds]])\n        name=\'Ultralisk27\' if mode==\'full\' else \'Ultralisk27Smoke\';extra=dict(cache_sha256=sha(cache),fingerprint=fp)\n        # Frozen calibration from all seven days; retained for diagnostics.\n        w,m=u[\'calibration\']({k:v[:nc] for k,v in p.items()},Y[ca],\'NarrowSpatial\',h[ca]);np.savez(out/\'ultra_calibration.npz\',weights=w,bias=m)\n    for day in days[2:]:\n        fit=np.flatnonzero(dt<day);val=np.flatnonzero(dt==day);base_oof.append(predict(fit,val));ii.append(val)\n    ids=np.concatenate(ii);return name,predict(np.arange(nc),np.arange(len(both))),ids,np.concatenate(base_oof),extra\n\ndef run_audit(args):\n    out=Path(args.out).resolve();out.mkdir(parents=True,exist_ok=True);data=data_root(args.data);dh=fingerprint(data);mode=args.mode;start=time.perf_counter()\n    train=readnp(data/\'train.npz\');s=readnp(data/\'static.npz\');Y=train[\'Y\'];T=train[\'times\'].astype(str);X,t,h=A[\'prepare\'](train[\'X\'],T,s)\n    assert len(T)==2935 and np.all(T[1:]>T[:-1]) and Y.shape==(2935,70,35,4)\n    tr=np.flatnonzero(T<CUT);ca=np.flatnonzero((T>=CUT)&(T<CAL)&(h%3==0));ev=np.flatnonzero((T>=CAL)&(T<END)&(h%3==0));assert (len(tr),len(ca),len(ev))==(1999,56,256)\n    if mode==\'smoke\':tr=tr[::12];ca=ca[::8];ev=ev[::8]\n    both=np.r_[ca,ev];nc=len(ca);base,bp,oid,boof,extra=baseline(data,out,mode,tr,ca,ev,T,h,Y,dh,args.cache);pred={base:bp};policies={};np.savez(out/\'static.npz\',**s)\n    phys=fit_anchor(X,t,h,Y,tr,s,True);np.savez(out/\'anchor_physics.npz\',**phys);ae=anchor_predict(phys,X[both],t[both],h[both]);pred[\'AtlasAnchor\']=ae\n    # Reload comparison verifies prediction path does not depend on training labels.\n    reloaded=anchor_predict(readnp(out/\'anchor_physics.npz\'),X[both],t[both],h[both]);assert np.max(np.abs(ae-reloaded))<=1e-4\n    raw=fit_anchor(X,t,h,Y,tr,s,False);np.savez(out/\'anchor_raw.npz\',**raw);ar=anchor_predict(raw,X[both],t[both],h[both]);pred[\'AtlasAnchorRaw\']=ar\n    if mode==\'proxy\':\n        prev=extra.pop(\'old_anchor\');extra[\'anchor_vs_previous_max_abs\']=float(np.max(np.abs(prev-ae)));extra[\'anchor_vs_previous_skill_delta\']=score(ae[nc:],Y[ev])-score(prev[nc:],Y[ev])\n    else:\n        devices=[torch.device(\'cpu\')] if mode==\'smoke\' else [torch.device(\'cuda:0\'),torch.device(\'cuda:1\')];width=16 if mode==\'smoke\' else 48;epochs=1 if mode==\'smoke\' else args.epochs;seeds=[42] if mode==\'smoke\' else SEEDS\n        ix,oo,man=crossfit(X,t,h,Y,T,tr,s,True,mode==\'smoke\');ir,orr,mr=crossfit(X,t,h,Y,T,tr,s,False,mode==\'smoke\');assert np.array_equal(ix,ir)\n        pd.DataFrame(man+mr).to_csv(out/\'crossfit_manifest.csv\',index=False);infit=anchor_predict(phys,X[ix],t[ix],h[ix]);extra[\'crossfit\']={\'rows\':len(ix),\'in_sample_skill\':score(infit,Y[ix]),\'oof_skill\':score(oo,Y[ix])}\n        for name,kind,at,ap in [(\'AtlasPhysicsInSample\',\'physics\',infit,ae),(\'AtlasPhysicsOOF\',\'physics\',oo,ae),(\'AtlasRawOOF\',\'raw\',orr,ar),(\'AtlasAuxOOF\',\'aux\',oo,ae),(\'AtlasWindOOF\',\'wind\',oo,ae)]:\n            paths=net_family(out,name,kind,X,t,Y,ix,at,s,devices,epochs,width,seeds);pred[name]=avg_predict(paths,X[both],t[both],s,ap,devices[0]);\n            # Replay from serialized EMA weights (same FP32 path).\n            p2=avg_predict(paths,X[both][:2],t[both][:2],s,ap[:2],devices[0]);assert np.allclose(pred[name][:2],p2,rtol=0,atol=1e-4),name\n        extra.update(epochs=epochs,width=width,trained_seeds=seeds)\n    specs={\'MixAnchor\':([\'AtlasAnchor\'],[False]),\'MixAnchorUV\':([\'AtlasAnchor\'],[True]),\'MixAnchorRaw\':([\'AtlasAnchorRaw\'],[False])}\n    if mode!=\'proxy\':specs.update({\'MixInSample\':([\'AtlasAnchor\',\'AtlasPhysicsInSample\'],[False,False]),\'MixPhysics\':([\'AtlasAnchor\',\'AtlasPhysicsOOF\'],[False,False]),\'MixRaw\':([\'AtlasAnchorRaw\',\'AtlasRawOOF\'],[False,False]),\'MixAux\':([\'AtlasAnchor\',\'AtlasAuxOOF\'],[False,False]),\'MixWind\':([\'AtlasAnchor\',\'AtlasWindOOF\'],[False,True]),\'MixFull\':([\'AtlasAnchor\',\'AtlasPhysicsOOF\',\'AtlasWindOOF\'],[False,False,True])})\n    for name,(components,flags) in specs.items():\n        w,report=nested(boof,[pred[k][oid] for k in components],Y[ca[oid]],daystr(T[ca[oid]]),flags);pred[name]=chain(bp,[pred[k] for k in components],w);policies[name]=dict(components=components,weights=[x.tolist() for x in w],**report)\n    correlations=[]\n    for period,ref,ids,target in [(\'prequential_calibration\',boof,oid,Y[ca[oid]]),(\'evaluation\',bp[nc:],np.arange(nc,len(both)),Y[ev])]:\n        for name in [\'AtlasAnchor\',\'AtlasAnchorRaw\',\'AtlasPhysicsOOF\',\'AtlasWindOOF\']:\n            if name not in pred:continue\n            e=ref.astype(\'f8\')-target;f=pred[name][ids].astype(\'f8\')-target\n            for j,v in enumerate(VARS):correlations.append(dict(period=period,reference=base,model=name,variable=v,pearson=float(np.corrcoef(e[...,j].ravel(),f[...,j].ravel())[0,1])))\n    pd.DataFrame(correlations).to_csv(out/\'error_correlations.csv\',index=False)\n    primary=\'MixAnchor\' if mode==\'proxy\' else \'MixFull\';rows,gates=evaluate({k:v[nc:] for k,v in pred.items()},Y[ev],T[ev],h[ev],t[ev,6:9],Y[tr],out,base,primary,mode)\n    np.savez(out/\'validation_predictions.npz\',times=T[both],n_cal=nc,target=Y[both],**pred)\n    report=dict(model=\'Atlas v3.2\',mode=mode,base=base,primary=primary,scope=\'retrospective reused historical labels; no blind score\',train_count=len(tr),calibration_count=nc,eval_count=len(ev),missing_days=sorted(set(pd.date_range(\'2024-05-28\',\'2024-06-30\').strftime(\'%Y%m%d\'))-set(daystr(T[ev]))),missing_history={k:float(np.mean(t[ix,6:8].sum(1)<2)) for k,ix in [(\'train\',tr),(\'cal\',ca),(\'eval\',ev)]},policy=policies,metrics=rows,gate=gates,review_eligible=any(g[\'review_eligible\'] for g in gates),recommend_submit=False,seconds=time.perf_counter()-start,extra=extra)\n    families={} if mode==\'proxy\' else {name:[f\'{name}_{seed}.pt\' for seed in seeds] for name in [\'AtlasPhysicsOOF\',\'AtlasWindOOF\']}\n    dump(out/\'bundle.json\',dict(model=\'Atlas v3.2 audit replay\',policy=policies[primary],networks=families,static=\'static.npz\',anchor=\'anchor_physics.npz\',base_sha256=BASE_HASH,training=\'frozen historical audit; NOT a final refit\'))\n    dump(out/\'report.json\',report);manifest(out,dh,mode,dict(primary=primary));print(pd.DataFrame(rows)[[\'model\',\'skill\']].to_string(index=False));print(\'REVIEW_ELIGIBLE\',report[\'review_eligible\'],\'; no automatic submission\')\n\ndef refit(args):\n    audit=Path(args.audit);data=data_root(args.data);dh=fingerprint(data);m=verify_manifest(audit,dh);r=json.loads((audit/\'report.json\').read_text())\n    if m[\'mode\']!=\'full\' or not r[\'review_eligible\'] or not args.approve:raise ValueError(\'Requires full audit gate AND explicit --approve; no automatic deployment\')\n    out=Path(args.out);out.mkdir(parents=True,exist_ok=True);train=readnp(data/\'train.npz\');s=readnp(data/\'static.npz\');T=train[\'times\'].astype(str);Y=train[\'Y\'];X,t,h=A[\'prepare\'](train[\'X\'],T,s);tr=np.arange(len(T));devices=[torch.device(\'cuda:0\'),torch.device(\'cuda:1\')]\n    if torch.cuda.device_count()!=2 or any(\'T4\' not in torch.cuda.get_device_name(i) for i in range(2)):raise RuntimeError(\'T4 x2 required\')\n    st=fit_anchor(X,t,h,Y,tr,s,True);np.savez(out/\'anchor_physics.npz\',**st);np.savez(out/\'static.npz\',**s);ix,oo,man=crossfit(X,t,h,Y,T,tr,s,True);pd.DataFrame(man).to_csv(out/\'crossfit_manifest.csv\',index=False)\n    policy=r[\'policy\'][\'MixFull\'];epochs=r[\'extra\'][\'epochs\'];families={}\n    for name,kind in [(\'AtlasPhysicsOOF\',\'physics\'),(\'AtlasWindOOF\',\'wind\')]:\n        paths=net_family(out,name,kind,X,t,Y,ix,oo,s,devices,epochs,48,SEEDS);families[name]=[p.name for p in paths]\n    dump(out/\'bundle.json\',dict(model=\'Atlas v3.2\',policy=policy,networks=families,static=\'static.npz\',anchor=\'anchor_physics.npz\',base_sha256=BASE_HASH,training=\'all official train rows; fixed audit blend weights; no test labels\',audit_manifest_sha256=sha(audit/\'manifest.json\')));manifest(out,dh,\'refit\',dict(audit_sha256=sha(audit/\'manifest.json\')));print(\'Bundle exported. Inference requires upstream Ultralisk forecast.\')\n\ndef infer(args):\n    bundle=Path(args.bundle);m=verify_manifest(bundle);\n    if args.base_csv and m[\'mode\']!=\'refit\':raise ValueError(\'No Kaggle CSV from audit or smoke bundle\')\n    b=json.loads((bundle/\'bundle.json\').read_text());s=readnp(bundle/b[\'static\']);\n    # Read only X and times; Y is neither accessed nor required.\n    with np.load(args.input,allow_pickle=False) as f:x=f[\'X\'];T=f[\'times\'].astype(str)\n    assert len(T)==len(x) and np.all(T[1:]>T[:-1]) and x.shape[1:]==(17,13,4)\n    X,t,h=A[\'prepare\'](x,T,s);keep=np.flatnonzero(h%3==0);X,t,h,T=X[keep],t[keep],h[keep],T[keep];ae=anchor_predict(readnp(bundle/b[\'anchor\']),X,t,h);device=torch.device(\'cuda:0\' if torch.cuda.is_available() else \'cpu\');pred={\'AtlasAnchor\':ae}\n    for k,paths in b[\'networks\'].items():pred[k]=avg_predict([bundle/p for p in paths],X,t,s,ae,device)\n    out=Path(args.out);out.mkdir(parents=True,exist_ok=True)\n    np.savez(out/\'atlas_predictions.npz\',times=T,**pred)\n    if args.base_csv:\n        if sha(args.base_csv)!=b[\'base_sha256\']:raise ValueError(\'Historical baseline CSV SHA-256 differs\')\n        if not args.template:raise ValueError(\'--template required for Kaggle CSV\')\n        template=pd.read_csv(args.template,dtype={\'id\':str});base=pd.read_csv(args.base_csv,dtype={\'id\':str});assert template.columns.tolist()==base.columns.tolist()==[\'id\',*VARS] and template.id.is_unique and base.id.is_unique and np.array_equal(template.id,base.id)\n        ids=[f\'{time}_{i:02d}_{j:02d}\' for time in T for i in range(70) for j in range(35)];assert len(ids)==646800 and set(ids)==set(base.id);p=base.set_index(\'id\').loc[ids,VARS].to_numpy(dtype=\'f4\').reshape(len(T),70,35,4)\n    elif args.base_npz:\n        z=readnp(args.base_npz);assert np.array_equal(z[\'times\'].astype(str),T);p=z[\'prediction\'];assert p.shape==ae.shape and np.isfinite(p).all()\n    else:print(\'Atlas-only predictions exported. Upstream Ultralisk required for ensemble.\');return\n    policy=b[\'policy\'];q=chain(p,[pred[k] for k in policy[\'components\']],policy[\'weights\']);np.savez(out/\'ensemble_predictions.npz\',times=T,prediction=q)\n    if args.base_csv:A[\'save_csv\'](template,T,q,out/\'submission_atlas_v3_2_REVIEW.csv\')\n    dump(out/\'inference_manifest.json\',dict(bundle_manifest_sha256=sha(bundle/\'manifest.json\'),input_sha256=sha(args.input),baseline_sha256=sha(args.base_csv or args.base_npz),outputs={p.name:sha(p) for p in out.iterdir() if p.is_file() and p.name!=\'inference_manifest.json\'},recommend_submit=False))\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--mode\',choices=[\'proxy\',\'smoke\',\'full\',\'refit\',\'infer\'],default=\'full\');p.add_argument(\'--data\',default=\'/kaggle/input\');p.add_argument(\'--out\',default=\'/kaggle/working/Atlas-v3.2\');p.add_argument(\'--cache\');p.add_argument(\'--epochs\',type=int,default=48);p.add_argument(\'--audit\');p.add_argument(\'--approve\',action=\'store_true\');p.add_argument(\'--bundle\');p.add_argument(\'--input\');p.add_argument(\'--base-csv\');p.add_argument(\'--base-npz\');p.add_argument(\'--template\');args=p.parse_args();torch.set_num_threads(2);torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True;torch.use_deterministic_algorithms(True,warn_only=True)\n    if args.mode==\'refit\':refit(args)\n    elif args.mode==\'infer\':infer(args)\n    else:run_audit(args)\n'
# Kaggle: Accelerator GPU T4 x2, Internet OFF. Dataset oficial adjunto.
from pathlib import Path
import subprocess, sys
SCRIPT=Path('/kaggle/working/Atlas_v3_2.py')
SCRIPT.write_text(SOURCE, encoding='utf-8')
MODE='full'  # full | smoke | proxy | refit | infer
DATA='/kaggle/input'
OUT='/kaggle/working/Atlas-v3.2-full'
# Usar un directorio nuevo por ejecución. Para reutilizar una caché del mismo corte:
CACHE=None
EXTRA=[]
# proxy: CACHE='/kaggle/input/.../validation_predictions.npz' (proxy v3.1)
# refit: EXTRA=['--audit','/kaggle/working/Atlas-v3.2-full','--approve']; cambiar OUT.
# infer: EXTRA=['--bundle','/kaggle/working/Atlas-v3.2-refit','--input','/kaggle/input/.../test.npz','--base-csv','/kaggle/input/.../submission_ultralisk_v2_7.csv','--template','/kaggle/input/.../sample_submission.csv']; cambiar OUT.
cmd=[sys.executable,str(SCRIPT),'--mode',MODE,'--data',DATA,'--out',OUT,'--epochs','48']
if CACHE:cmd+=['--cache',CACHE]
subprocess.run(cmd+EXTRA,check=True)
